# Notebook 04 — Course, Jurisdiction and Surface Mapping

## Purpose

This notebook investigates how racecourses, jurisdictions and racing surfaces are represented across the source data, and which canonical attributes can be derived reliably without discarding jurisdiction-specific meaning.

The raw `course` field contains venue names and, for many international courses, apparent jurisdiction suffixes or other naming conventions. Surface information may be represented through several fields, including:

- `course`;
- `race_name`;
- `type`;
- combinations of those fields.

These representations have not yet been tested systematically. Course names may also vary across time, distinguish multiple tracks at one physical venue, or appear similar despite referring to venues in different jurisdictions.

This notebook will test:

- the inventory and frequency of raw `course` values;
- how course names encode country or racing jurisdiction;
- which suffixes and naming patterns can support jurisdiction parsing;
- where jurisdiction cannot be inferred reliably from `course` alone;
- how turf, all-weather and other surface information is represented;
- whether surface evidence appears in `course`, `race_name`, `type`, or combinations of fields;
- where one apparent physical venue has multiple raw course-name variants;
- where similar course names may refer to different jurisdictions or materially different tracks;
- how course names and representations vary over time;
- which raw values and source-lineage attributes must be preserved;
- which candidate canonical course, jurisdiction and surface attributes a later staging layer may require.

The notebook will produce course-name inventories, parsing evidence, candidate canonicalisation rules, collision and ambiguity evidence, and unresolved-risk recommendations. It will not design the final target schema.

In [1]:
# Read-only source setup for Notebook 04.
#
# The raw SQLite database is immutable. Opening it with SQLite's URI
# read-only mode prevents this notebook from modifying the source file.

from pathlib import Path
import sqlite3

# Resolve the repository root whether the notebook is run from the project
# root or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_DB = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"

if not SOURCE_DB.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB}")

# mode=ro enforces read-only access at the SQLite connection level.
connection = sqlite3.connect(f"file:{SOURCE_DB}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row

print(f"Project root: {PROJECT_ROOT}")
print(f"Source database: {SOURCE_DB}")
print("SQLite access: read-only")
print(f"Source table: {SOURCE_TABLE}")
print(f"Data-row predicate: {DATA_ROW_PREDICATE}")

Project root: /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
SQLite access: read-only
Source table: data
Data-row predicate: rowid <> 1


In [2]:
# Measure the headline structure of the raw course field.
#
# This cell establishes:
# - total data-like runner rows;
# - number of distinct raw course values;
# - whether blank course values exist;
# - minimum and maximum dates attached to the course population.
#
# This is descriptive only. It does not yet attempt to parse jurisdictions,
# identify physical venues, or canonicalise course names.

import pandas as pd

course_summary_sql = f"""
SELECT
    COUNT(*) AS runner_rows,
    COUNT(DISTINCT course) AS distinct_raw_course_values,
    SUM(CASE WHEN TRIM(CAST(course AS TEXT)) = '' THEN 1 ELSE 0 END)
        AS blank_course_rows,
    MIN(date) AS earliest_date,
    MAX(date) AS latest_date
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
"""

course_summary = pd.read_sql_query(course_summary_sql, connection)
course_summary

,runner_rows,distinct_raw_course_values,blank_course_rows,earliest_date,latest_date
0,1851285,528,0,2015-01-01,2026-05-27


In [3]:
# Build the raw course-name inventory.
#
# For each exact source value, this cell measures:
# - runner-row frequency;
# - provisional race frequency using date + course + off;
# - number of active dates;
# - first and last observed dates.
#
# Raw course values are left unchanged. No trimming, suffix removal,
# jurisdiction parsing, or canonicalisation is applied here.

course_inventory_sql = f"""
WITH provisional_races AS (
    SELECT
        date,
        course,
        off
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off
),
race_counts AS (
    SELECT
        course,
        COUNT(*) AS provisional_races
    FROM provisional_races
    GROUP BY course
),
runner_counts AS (
    SELECT
        course,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT date) AS active_dates,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY course
)
SELECT
    rc.course,
    rc.runner_rows,
    races.provisional_races,
    rc.active_dates,
    rc.first_date,
    rc.last_date
FROM runner_counts AS rc
JOIN race_counts AS races
    ON races.course = rc.course
ORDER BY
    rc.runner_rows DESC,
    rc.course
"""

course_inventory = pd.read_sql_query(course_inventory_sql, connection)

print(f"Inventory rows: {len(course_inventory):,}")
course_inventory.head(30)

Inventory rows: 528


,course,runner_rows,provisional_races,active_dates,first_date,last_date
0,Wolverhampton (AW),64746,7042,930,2015-01-02,2026-05-18
1,Sha Tin (HK),52010,4160,435,2015-01-25,2025-10-12
2,Kempton (AW),50268,5046,662,2015-01-07,2026-05-27
3,Chantilly (FR),48744,4140,497,2015-01-23,2025-10-14
4,Deauville (FR),48618,4051,472,2015-01-03,2025-08-30
5,Newcastle (AW),42092,4409,583,2016-05-17,2026-05-19
6,Lingfield (AW),41632,4803,742,2015-01-03,2026-05-12
7,Dundalk (AW) (IRE),39075,3318,428,2015-01-02,2025-10-14
8,Chelmsford (AW),37741,4310,593,2015-01-11,2026-03-26
9,Southwell (AW),34488,3816,515,2015-01-01,2026-05-21


In [4]:
# Inventory the parenthetical patterns used in raw course names.
#
# This cell extracts the complete trailing parenthetical portion of each
# distinct raw course value. Examples include:
# - no suffix;
# - (AW);
# - (IRE);
# - (AW) (IRE);
# - (FR).
#
# The output measures both:
# - how many distinct raw course values use each pattern;
# - how many runner rows and provisional races those values represent.
#
# This remains descriptive. No pattern is yet accepted as a jurisdiction
# or surface rule.

import re

course_suffix_inventory = course_inventory.copy()

def extract_parenthetical_pattern(course_name):
    """Return all parenthetical groups in their original order."""
    groups = re.findall(r"\([^()]+\)", str(course_name))
    return " ".join(groups) if groups else "[none]"

course_suffix_inventory["parenthetical_pattern"] = (
    course_suffix_inventory["course"].map(extract_parenthetical_pattern)
)

parenthetical_pattern_summary = (
    course_suffix_inventory
    .groupby("parenthetical_pattern", as_index=False)
    .agg(
        distinct_course_values=("course", "nunique"),
        runner_rows=("runner_rows", "sum"),
        provisional_races=("provisional_races", "sum"),
    )
    .sort_values(
        ["runner_rows", "distinct_course_values", "parenthetical_pattern"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Distinct parenthetical patterns: "
    f"{len(parenthetical_pattern_summary):,}"
)

parenthetical_pattern_summary

Distinct parenthetical patterns: 44


,parenthetical_pattern,distinct_course_values,runner_rows,provisional_races
0,[none],189,749748,84895
1,(IRE),25,299546,25882
2,(AW),7,274133,29705
3,(FR),73,214637,19361
4,(HK),2,83068,6847
5,(USA),55,48156,5860
6,(AUS),50,42879,3767
7,(AW) (IRE),1,39075,3318
8,(UAE),5,25178,2219
9,(JPN),21,21202,1470


In [5]:
# Inspect raw course values whose parentheses may not represent jurisdiction alone.
#
# We include:
# - values containing more than one parenthetical group;
# - recognised surface or course-configuration markers;
# - geographic qualifiers that also appear without a jurisdiction suffix.
#
# This targeted inspection will help separate:
# - jurisdiction markers;
# - surface markers;
# - venue or track variants;
# - other geographic qualifiers.
#
# No canonical names or parsing rules are assigned yet.

patterns_for_inspection = {
    "(AW)",
    "(AW) (IRE)",
    "(July)",
    "(RH) (IRE)",
    "(Perth)",
    "(Perth) (AUS)",
    "(South Carolina)",
    "(South Carolina) (USA)",
}

special_parenthetical_courses = (
    course_suffix_inventory.loc[
        course_suffix_inventory["parenthetical_pattern"].isin(
            patterns_for_inspection
        ),
        [
            "course",
            "parenthetical_pattern",
            "runner_rows",
            "provisional_races",
            "active_dates",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["parenthetical_pattern", "runner_rows", "course"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

special_parenthetical_courses

,course,parenthetical_pattern,runner_rows,provisional_races,active_dates,first_date,last_date
0,Wolverhampton (AW),(AW),64746,7042,930,2015-01-02,2026-05-18
1,Kempton (AW),(AW),50268,5046,662,2015-01-07,2026-05-27
2,Newcastle (AW),(AW),42092,4409,583,2016-05-17,2026-05-19
3,Lingfield (AW),(AW),41632,4803,742,2015-01-03,2026-05-12
4,Chelmsford (AW),(AW),37741,4310,593,2015-01-11,2026-03-26
5,Southwell (AW),(AW),34488,3816,515,2015-01-01,2026-05-21
6,Dundalk (AW),(AW),3166,279,37,2025-10-17,2026-05-26
7,Dundalk (AW) (IRE),(AW) (IRE),39075,3318,428,2015-01-02,2025-10-14
8,Newmarket (July),(July),11820,1438,208,2015-06-19,2025-08-23
9,Belmont Park (Perth),(Perth),10,1,1,2026-05-16,2026-05-16


In [6]:
# Identify apparent course variants after removing only a recognised terminal
# jurisdiction code.
#
# Examples:
# - "Dundalk (AW) (IRE)" becomes "Dundalk (AW)";
# - "Belmont Park (Perth) (AUS)" becomes "Belmont Park (Perth)";
# - "Newmarket (July)" remains unchanged because July is not jurisdiction;
# - "Wolverhampton (AW)" remains unchanged because AW is a surface marker.
#
# The derived comparison label is used only to locate possible variants.
# It does not replace or alter the raw course value.

recognised_terminal_jurisdiction_codes = {
    "ARG", "AUS", "BEL", "BHR", "BRZ", "CAN", "CHI", "CHN",
    "CZE", "DEN", "FR", "GER", "GUE", "HK", "HUN", "IRE",
    "ITY", "JER", "JPN", "KOR", "KSA", "NOR", "NZ", "PER",
    "POL", "QA", "SAF", "SIN", "SPA", "SWE", "SWI", "TUR",
    "UAE", "URU", "USA",
}

terminal_code_pattern = re.compile(r"\s+\(([^()]+)\)$")

def remove_recognised_terminal_jurisdiction(course_name):
    """Remove the final parenthetical group only when it is a known code."""
    raw_value = str(course_name)
    match = terminal_code_pattern.search(raw_value)

    if match and match.group(1) in recognised_terminal_jurisdiction_codes:
        return raw_value[:match.start()]

    return raw_value

course_variant_inventory = course_inventory.copy()
course_variant_inventory["comparison_course_label"] = (
    course_variant_inventory["course"].map(
        remove_recognised_terminal_jurisdiction
    )
)

variant_group_sizes = (
    course_variant_inventory
    .groupby("comparison_course_label")["course"]
    .nunique()
)

multi_value_comparison_labels = variant_group_sizes[
    variant_group_sizes > 1
].index

apparent_course_variants = (
    course_variant_inventory.loc[
        course_variant_inventory["comparison_course_label"].isin(
            multi_value_comparison_labels
        ),
        [
            "comparison_course_label",
            "course",
            "runner_rows",
            "provisional_races",
            "active_dates",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["comparison_course_label", "first_date", "course"]
    )
    .reset_index(drop=True)
)

print(
    "Comparison labels with multiple raw course values: "
    f"{len(multi_value_comparison_labels):,}"
)
print(
    "Raw course values inside those groups: "
    f"{len(apparent_course_variants):,}"
)

apparent_course_variants

Comparison labels with multiple raw course values: 136
Raw course values inside those groups: 272


,comparison_course_label,course,runner_rows,provisional_races,active_dates,first_date,last_date
0,Abu Dhabi,Abu Dhabi (UAE),2002,152,137,2015-01-04,2025-04-12
1,Abu Dhabi,Abu Dhabi,427,32,16,2025-10-25,2026-04-11
2,Aqueduct,Aqueduct (USA),2204,297,181,2015-01-01,2025-04-05
3,Aqueduct,Aqueduct,199,30,14,2025-10-18,2026-05-09
4,Ascot,Ascot (AUS),2407,204,140,2015-01-01,2025-05-10
...,...,...,...,...,...,...,...
267,Vichy,Vichy,9,1,1,2026-05-13,2026-05-13
268,Wexford,Wexford (IRE),8539,789,109,2015-04-21,2025-08-29
269,Wexford,Wexford,541,52,7,2025-10-26,2026-05-27
270,Woodbine,Woodbine (CAN),3519,434,283,2015-04-18,2025-10-11


In [7]:
# Test whether raw course variants sharing a comparison label coexist
# on the same racing date.
#
# A group such as:
# - Dundalk (AW) (IRE)
# - Dundalk (AW)
#
# may represent a source naming transition if the forms occur in separate
# periods.
#
# By contrast, same-date use of multiple raw values under one comparison
# label is strong evidence that removing the jurisdiction suffix creates
# a collision between distinct venues.
#
# This cell reports group-level evidence only. It does not yet classify
# or canonicalise any individual course.

variant_raw_values = apparent_course_variants["course"].tolist()

# Build placeholders safely rather than interpolating course names into SQL.
placeholders = ", ".join("?" for _ in variant_raw_values)

variant_dates_sql = f"""
SELECT DISTINCT
    date,
    course
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND course IN ({placeholders})
"""

variant_dates = pd.read_sql_query(
    variant_dates_sql,
    connection,
    params=variant_raw_values,
)

# Attach the provisional comparison label created in the previous cell.
course_to_comparison_label = dict(
    zip(
        course_variant_inventory["course"],
        course_variant_inventory["comparison_course_label"],
    )
)

variant_dates["comparison_course_label"] = (
    variant_dates["course"].map(course_to_comparison_label)
)

# Count how many different raw course values occur for each label and date.
label_date_counts = (
    variant_dates
    .groupby(
        ["comparison_course_label", "date"],
        as_index=False,
    )
    .agg(raw_course_values_on_date=("course", "nunique"))
)

same_date_collisions = (
    label_date_counts.loc[
        label_date_counts["raw_course_values_on_date"] > 1
    ]
    .groupby("comparison_course_label", as_index=False)
    .agg(
        dates_with_multiple_raw_values=("date", "nunique"),
        first_collision_date=("date", "min"),
        last_collision_date=("date", "max"),
        maximum_raw_values_on_one_date=(
            "raw_course_values_on_date",
            "max",
        ),
    )
)

variant_group_summary = (
    apparent_course_variants
    .groupby("comparison_course_label", as_index=False)
    .agg(
        distinct_raw_course_values=("course", "nunique"),
        combined_runner_rows=("runner_rows", "sum"),
        first_observed_date=("first_date", "min"),
        last_observed_date=("last_date", "max"),
    )
    .merge(
        same_date_collisions,
        on="comparison_course_label",
        how="left",
    )
)

variant_group_summary["dates_with_multiple_raw_values"] = (
    variant_group_summary["dates_with_multiple_raw_values"]
    .fillna(0)
    .astype(int)
)

variant_group_summary["has_same_date_collision"] = (
    variant_group_summary["dates_with_multiple_raw_values"] > 0
)

variant_group_summary = (
    variant_group_summary
    .sort_values(
        [
            "has_same_date_collision",
            "dates_with_multiple_raw_values",
            "combined_runner_rows",
            "comparison_course_label",
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Comparison labels tested: "
    f"{len(variant_group_summary):,}"
)
print(
    "Labels with multiple raw forms on at least one same date: "
    f"{variant_group_summary['has_same_date_collision'].sum():,}"
)

variant_group_summary.head(30)

Comparison labels tested: 136
Labels with multiple raw forms on at least one same date: 3


,comparison_course_label,distinct_raw_course_values,combined_runner_rows,first_observed_date,last_observed_date,dates_with_multiple_raw_values,first_collision_date,last_collision_date,maximum_raw_values_on_one_date,has_same_date_collision
0,Ascot,2,22076,2015-01-01,2026-05-09,39,2015-10-31,2025-05-10,2.0,True
1,Sandown,2,14716,2015-01-03,2026-04-25,2,2019-07-24,2025-02-01,2.0,True
2,Newcastle,2,9320,2015-01-03,2026-04-11,2,2015-03-14,2015-04-11,2.0,True
3,Sha Tin,2,56963,2015-01-25,2026-05-24,0,NaN,NaN,NaN,False
4,Chantilly,2,52403,2015-01-23,2026-05-19,0,NaN,NaN,NaN,False
5,Deauville,2,52014,2015-01-03,2026-04-08,0,NaN,NaN,NaN,False
6,Dundalk (AW),2,42241,2015-01-02,2026-05-26,0,NaN,NaN,NaN,False
7,Auteuil,2,34928,2015-03-01,2026-05-26,0,NaN,NaN,NaN,False
8,Happy Valley,2,34012,2016-01-06,2026-05-27,0,NaN,NaN,NaN,False
9,Saint-Cloud,2,30288,2015-03-07,2026-05-22,0,NaN,NaN,NaN,False


In [8]:
# Inspect the exact raw course values and collision dates for the three
# comparison labels that coexist on the same date.
#
# The same list of course values is used in two SQL IN clauses, so the
# parameter list must also be supplied twice.

colliding_labels = (
    variant_group_summary.loc[
        variant_group_summary["has_same_date_collision"],
        "comparison_course_label",
    ]
    .tolist()
)

collision_course_values = (
    apparent_course_variants.loc[
        apparent_course_variants["comparison_course_label"].isin(
            colliding_labels
        ),
        "course",
    ]
    .tolist()
)

course_placeholders = ", ".join("?" for _ in collision_course_values)

collision_detail_sql = f"""
WITH collision_dates AS (
    SELECT
        date,
        CASE
            WHEN course IN ('Ascot', 'Ascot (AUS)') THEN 'Ascot'
            WHEN course IN ('Sandown', 'Sandown (AUS)') THEN 'Sandown'
            WHEN course IN ('Newcastle', 'Newcastle (AUS)') THEN 'Newcastle'
        END AS comparison_course_label
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND course IN ({course_placeholders})
    GROUP BY
        date,
        course
),
confirmed_collision_dates AS (
    SELECT
        comparison_course_label,
        date
    FROM collision_dates
    GROUP BY
        comparison_course_label,
        date
    HAVING COUNT(*) > 1
)
SELECT
    CASE
        WHEN d.course IN ('Ascot', 'Ascot (AUS)') THEN 'Ascot'
        WHEN d.course IN ('Sandown', 'Sandown (AUS)') THEN 'Sandown'
        WHEN d.course IN ('Newcastle', 'Newcastle (AUS)') THEN 'Newcastle'
    END AS comparison_course_label,
    d.date,
    d.course,
    COUNT(*) AS runner_rows,
    COUNT(
        DISTINCT
        d.date || '|' || d.course || '|' || CAST(d.off AS TEXT)
    ) AS provisional_races
FROM {SOURCE_TABLE} AS d
JOIN confirmed_collision_dates AS c
    ON c.date = d.date
   AND c.comparison_course_label =
       CASE
           WHEN d.course IN ('Ascot', 'Ascot (AUS)') THEN 'Ascot'
           WHEN d.course IN ('Sandown', 'Sandown (AUS)') THEN 'Sandown'
           WHEN d.course IN ('Newcastle', 'Newcastle (AUS)') THEN 'Newcastle'
       END
WHERE {DATA_ROW_PREDICATE}
  AND d.course IN ({course_placeholders})
GROUP BY
    comparison_course_label,
    d.date,
    d.course
ORDER BY
    comparison_course_label,
    d.date,
    d.course
"""

collision_details = pd.read_sql_query(
    collision_detail_sql,
    connection,
    params=collision_course_values + collision_course_values,
)

print(f"Collision-detail rows: {len(collision_details):,}")
collision_details

Collision-detail rows: 86


,comparison_course_label,date,course,runner_rows,provisional_races
0,Ascot,2015-10-31,Ascot,81,7
1,Ascot,2015-10-31,Ascot (AUS),8,1
2,Ascot,2015-11-21,Ascot,53,7
3,Ascot,2015-11-21,Ascot (AUS),45,3
4,Ascot,2015-12-19,Ascot,73,6
...,...,...,...,...,...
81,Newcastle,2015-04-11,Newcastle (AUS),38,3
82,Sandown,2019-07-24,Sandown,50,6
83,Sandown,2019-07-24,Sandown (AUS),10,1
84,Sandown,2025-02-01,Sandown,51,7


In [9]:
# Measure temporal handovers among comparison labels with no same-date collision.
#
# For each label, this cell identifies:
# - the earlier-ending raw course form;
# - the later-starting raw course form;
# - the last date of the earlier form;
# - the first date of the later form;
# - the calendar gap between those observations.
#
# A short positive gap across many labels would support a coordinated source
# naming change. It would not, by itself, prove that every pair is the same
# physical venue.

non_collision_variants = apparent_course_variants.loc[
    ~apparent_course_variants["comparison_course_label"].isin(
        variant_group_summary.loc[
            variant_group_summary["has_same_date_collision"],
            "comparison_course_label",
        ]
    )
].copy()

# Each current group contains two raw values. Sort them by their first
# observation so that the older and newer forms can be compared directly.
ordered_variants = (
    non_collision_variants
    .sort_values(
        ["comparison_course_label", "first_date", "last_date", "course"]
    )
    .reset_index(drop=True)
)

handover_rows = []

for comparison_label, group in ordered_variants.groupby(
    "comparison_course_label",
    sort=True,
):
    group = group.reset_index(drop=True)

    earlier = group.iloc[0]
    later = group.iloc[-1]

    earlier_last_date = pd.to_datetime(earlier["last_date"])
    later_first_date = pd.to_datetime(later["first_date"])

    handover_rows.append(
        {
            "comparison_course_label": comparison_label,
            "earlier_raw_course": earlier["course"],
            "earlier_last_date": earlier["last_date"],
            "later_raw_course": later["course"],
            "later_first_date": later["first_date"],
            "calendar_gap_days": (
                later_first_date - earlier_last_date
            ).days,
        }
    )

course_handover_summary = pd.DataFrame(handover_rows)

print(
    "Non-colliding comparison labels assessed: "
    f"{len(course_handover_summary):,}"
)
print()
print("Calendar-gap distribution:")
print(
    course_handover_summary["calendar_gap_days"]
    .value_counts()
    .sort_index()
    .to_string()
)

course_handover_summary.sort_values(
    ["later_first_date", "comparison_course_label"]
).head(40)

Non-colliding comparison labels assessed: 133

Calendar-gap distribution:
calendar_gap_days
1        1
3        1
4        1
5        1
6        3
7        6
10       1
12       1
13       1
14       3
16       1
18       1
19       1
21       3
23       1
24       1
26       2
28       7
29       1
32       1
33       1
35       1
40       1
41       1
42       1
45       1
47       1
51       1
53       1
58       1
59       2
69       2
70       2
81       1
84       1
90       1
98       2
118      1
126      1
133      1
146      1
154      1
170      1
175      1
185      1
189      4
191      1
196      3
202      1
204      1
210      1
215      1
216      1
217      1
219      1
226      1
230      1
231      1
235      1
238      1
239      1
245      1
252      2
253      1
258      2
280      1
287      1
292      1
299      1
308      1
315      2
329      1
350      1
357      3
364     16
365      2
371      2
447      1
685      1
700      1
728      1
731      1
1816  

,comparison_course_label,earlier_raw_course,earlier_last_date,later_raw_course,later_first_date,calendar_gap_days
14,Caulfield,Caulfield (AUS),2025-10-11,Caulfield,2025-10-15,4
53,Happy Valley,Happy Valley (HK),2025-10-08,Happy Valley,2025-10-15,7
91,Nantes,Nantes (FR),2025-04-09,Nantes,2025-10-15,189
101,Punchestown,Punchestown (IRE),2025-10-14,Punchestown,2025-10-15,1
25,Curragh,Curragh (IRE),2025-10-04,Curragh,2025-10-16,12
107,Saint-Cloud,Saint-Cloud (FR),2025-10-09,Saint-Cloud,2025-10-16,7
122,Thurles,Thurles (IRE),2025-10-09,Thurles,2025-10-16,7
3,Baden-Baden,Baden-Baden (GER),2025-09-07,Baden-Baden,2025-10-17,40
33,Dundalk (AW),Dundalk (AW) (IRE),2025-10-14,Dundalk (AW),2025-10-17,3
59,Keeneland,Keeneland (USA),2025-10-12,Keeneland,2025-10-17,5


In [10]:
# Quantify when the later raw course forms first appear.
#
# This tests whether the apparent suffix-removal transitions are concentrated
# around one source-wide change date rather than scattered independently.
#
# We summarise:
# - the first-observation month of each later form;
# - how many later forms began from 15 October 2025 onward;
# - how many earlier forms ended before their later form began.
#
# This remains evidence about source representation, not final
# course canonicalisation.

course_handover_summary["later_first_date_parsed"] = pd.to_datetime(
    course_handover_summary["later_first_date"]
)
course_handover_summary["earlier_last_date_parsed"] = pd.to_datetime(
    course_handover_summary["earlier_last_date"]
)

course_handover_summary["later_first_month"] = (
    course_handover_summary["later_first_date_parsed"]
    .dt.to_period("M")
    .astype(str)
)

handover_month_summary = (
    course_handover_summary
    .groupby("later_first_month", as_index=False)
    .agg(
        comparison_labels=("comparison_course_label", "nunique"),
        earliest_new_form=("later_first_date", "min"),
        latest_new_form=("later_first_date", "max"),
    )
    .sort_values("later_first_month")
    .reset_index(drop=True)
)

change_window_start = pd.Timestamp("2025-10-15")

headline_handover_evidence = pd.DataFrame(
    {
        "measure": [
            "non-colliding comparison labels",
            "later forms first seen from 2025-10-15 onward",
            "later forms first seen during October 2025",
            "pairs with no temporal overlap",
            "pairs with a gap of 31 days or fewer",
        ],
        "value": [
            len(course_handover_summary),
            (
                course_handover_summary["later_first_date_parsed"]
                >= change_window_start
            ).sum(),
            (
                course_handover_summary["later_first_month"]
                == "2025-10"
            ).sum(),
            (
                course_handover_summary["calendar_gap_days"] > 0
            ).sum(),
            (
                course_handover_summary["calendar_gap_days"]
                .between(1, 31)
            ).sum(),
        ],
    }
)

display(headline_handover_evidence)
handover_month_summary

,measure,value
0,non-colliding comparison labels,133
1,later forms first seen from 2025-10-15 onward,133
2,later forms first seen during October 2025,47
3,pairs with no temporal overlap,133
4,pairs with a gap of 31 days or fewer,37


,later_first_month,comparison_labels,earliest_new_form,latest_new_form
0,2025-10,47,2025-10-15,2025-10-31
1,2025-11,28,2025-11-01,2025-11-29
2,2025-12,14,2025-12-06,2025-12-29
3,2026-01,7,2026-01-05,2026-01-31
4,2026-02,11,2026-02-01,2026-02-25
5,2026-03,5,2026-03-08,2026-03-29
6,2026-04,3,2026-04-11,2026-04-24
7,2026-05,18,2026-05-02,2026-05-26


In [11]:
# Build provisional historical jurisdiction evidence for unsuffixed course forms.
#
# For each non-colliding comparison label, this cell:
# - identifies the terminal jurisdiction code from the earlier raw value;
# - links it to the later unsuffixed value;
# - measures how many later runner rows and races would inherit that evidence.
#
# This is not yet an accepted parsing rule. It tests the coverage and
# consistency of using historical course-name forms as a jurisdiction lookup.

def extract_terminal_jurisdiction_code(course_name):
    """Return a recognised terminal jurisdiction code, otherwise None."""
    match = terminal_code_pattern.search(str(course_name))

    if match and match.group(1) in recognised_terminal_jurisdiction_codes:
        return match.group(1)

    return None


historical_jurisdiction_links = course_handover_summary.copy()

historical_jurisdiction_links["historical_jurisdiction_code"] = (
    historical_jurisdiction_links["earlier_raw_course"].map(
        extract_terminal_jurisdiction_code
    )
)

# Attach frequencies for the later raw form.
later_form_frequencies = (
    course_inventory[
        [
            "course",
            "runner_rows",
            "provisional_races",
            "active_dates",
        ]
    ]
    .rename(
        columns={
            "course": "later_raw_course",
            "runner_rows": "later_runner_rows",
            "provisional_races": "later_provisional_races",
            "active_dates": "later_active_dates",
        }
    )
)

historical_jurisdiction_links = (
    historical_jurisdiction_links
    .merge(
        later_form_frequencies,
        on="later_raw_course",
        how="left",
    )
)

jurisdiction_link_summary = (
    historical_jurisdiction_links
    .groupby(
        "historical_jurisdiction_code",
        dropna=False,
        as_index=False,
    )
    .agg(
        comparison_labels=("comparison_course_label", "nunique"),
        later_raw_course_values=("later_raw_course", "nunique"),
        later_runner_rows=("later_runner_rows", "sum"),
        later_provisional_races=("later_provisional_races", "sum"),
    )
    .sort_values(
        ["later_runner_rows", "comparison_labels"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(
    "Historical links assessed: "
    f"{len(historical_jurisdiction_links):,}"
)
print(
    "Links with a recognised historical jurisdiction code: "
    f"{historical_jurisdiction_links['historical_jurisdiction_code'].notna().sum():,}"
)
print(
    "Later runner rows covered by recognised historical codes: "
    f"{historical_jurisdiction_links.loc[
        historical_jurisdiction_links['historical_jurisdiction_code'].notna(),
        'later_runner_rows'
    ].sum():,}"
)

jurisdiction_link_summary

Historical links assessed: 133
Links with a recognised historical jurisdiction code: 133
Later runner rows covered by recognised historical codes: 50,678


,historical_jurisdiction_code,comparison_labels,later_raw_course_values,later_runner_rows,later_provisional_races
0,IRE,23,23,18133,1568
1,FR,19,19,13695,1172
2,HK,2,2,7907,634
3,UAE,4,4,2712,232
4,AUS,21,21,2677,237
5,USA,21,21,1963,242
6,JPN,9,9,1349,89
7,BHR,1,1,593,59
8,ARG,3,3,270,21
9,NZ,5,5,217,19


In [12]:
# Inventory raw course values with no parenthetical markers that are not
# covered by a safe historical jurisdiction link.
#
# This isolates the remaining unsuffixed population after excluding:
# - later forms linked to an earlier suffixed course;
# - raw values containing any parenthetical marker.
#
# The result will include established British courses, but may also contain:
# - international courses without historical suffix evidence;
# - ambiguous names shared across jurisdictions;
# - newly introduced source values.
#
# No default jurisdiction is assigned in this cell.

historically_linked_later_courses = set(
    historical_jurisdiction_links["later_raw_course"]
)

unsuffixed_course_inventory = course_inventory.loc[
    ~course_inventory["course"].str.contains(r"\([^()]+\)", regex=True)
].copy()

unlinked_unsuffixed_courses = (
    unsuffixed_course_inventory.loc[
        ~unsuffixed_course_inventory["course"].isin(
            historically_linked_later_courses
        )
    ]
    .sort_values(
        ["runner_rows", "course"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

unlinked_unsuffixed_summary = pd.DataFrame(
    {
        "measure": [
            "all unsuffixed raw course values",
            "historically linked unsuffixed values",
            "remaining unlinked unsuffixed values",
            "runner rows in remaining values",
            "provisional races in remaining values",
        ],
        "value": [
            len(unsuffixed_course_inventory),
            len(
                unsuffixed_course_inventory.loc[
                    unsuffixed_course_inventory["course"].isin(
                        historically_linked_later_courses
                    )
                ]
            ),
            len(unlinked_unsuffixed_courses),
            unlinked_unsuffixed_courses["runner_rows"].sum(),
            unlinked_unsuffixed_courses["provisional_races"].sum(),
        ],
    }
)

display(unlinked_unsuffixed_summary)
unlinked_unsuffixed_courses.head(50)

,measure,value
0,all unsuffixed raw course values,189
1,historically linked unsuffixed values,130
2,remaining unlinked unsuffixed values,59
3,runner rows in remaining values,702255
4,provisional races in remaining values,80795


,course,runner_rows,provisional_races,active_dates,first_date,last_date
0,Doncaster,26166,2805,389,2015-01-09,2026-05-16
1,Newbury,21535,2223,310,2015-01-14,2026-05-16
2,Ayr,20921,2325,328,2015-01-02,2026-05-20
3,Chepstow,20804,2318,326,2015-01-30,2026-05-21
4,Ascot,19669,1837,284,2015-01-17,2026-05-09
5,Haydock,19065,2269,331,2015-01-17,2026-05-23
6,Windsor,17677,1935,274,2015-04-13,2026-05-25
7,Catterick,17389,2010,287,2015-01-01,2026-05-21
8,Musselburgh,16365,2007,290,2015-01-01,2026-05-21
9,Uttoxeter,16185,1833,256,2015-01-24,2026-05-24


In [13]:
# Display the complete inventory of unsuffixed course values that have no
# safe historical jurisdiction link.
#
# The earlier output showed only the first 50 of 59 values. This cell exposes
# the entire remaining population so that low-frequency exceptions are not
# hidden by the dominant British-course values.
#
# This is still observation only. No jurisdiction is assigned.

pd.set_option("display.max_rows", None)

unlinked_unsuffixed_courses[
    [
        "course",
        "runner_rows",
        "provisional_races",
        "active_dates",
        "first_date",
        "last_date",
    ]
]

,course,runner_rows,provisional_races,active_dates,first_date,last_date
0,Doncaster,26166,2805,389,2015-01-09,2026-05-16
1,Newbury,21535,2223,310,2015-01-14,2026-05-16
2,Ayr,20921,2325,328,2015-01-02,2026-05-20
3,Chepstow,20804,2318,326,2015-01-30,2026-05-21
4,Ascot,19669,1837,284,2015-01-17,2026-05-09
5,Haydock,19065,2269,331,2015-01-17,2026-05-23
6,Windsor,17677,1935,274,2015-04-13,2026-05-25
7,Catterick,17389,2010,287,2015-01-01,2026-05-21
8,Musselburgh,16365,2007,290,2015-01-01,2026-05-21
9,Uttoxeter,16185,1833,256,2015-01-24,2026-05-24


In [14]:
# Inspect the isolated unsuffixed Firenze course value.
#
# Firenze is the only value outside the established British-course population
# among the 59 unsuffixed courses without a historical jurisdiction link.
#
# This cell preserves the source values and displays the complete runner-level
# race context needed to assess whether jurisdiction or surface evidence appears
# elsewhere in the record.

firenze_rows_sql = f"""
SELECT
    rowid AS source_rowid,
    date,
    race_id,
    course,
    off,
    race_name,
    type,
    horse,
    pos,
    draw,
    ran,
    time,
    comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND course = 'Firenze'
ORDER BY
    date,
    off,
    source_rowid
"""

firenze_rows = pd.read_sql_query(firenze_rows_sql, connection)

print(f"Firenze runner rows: {len(firenze_rows):,}")
print(
    "Firenze provisional races: "
    f"{firenze_rows[['date', 'course', 'off']].drop_duplicates().shape[0]:,}"
)

firenze_rows

Firenze runner rows: 4
Firenze provisional races: 1


,source_rowid,date,race_id,course,off,race_name,type,horse,pos,draw,ran,time,comment
0,1833828,2026-04-25,919235,Firenze,16:15,Premio Natale di Roma (Turf),Flat,Kabir (IRE),1,1,4,1:30.10,
1,1833829,2026-04-25,919235,Firenze,16:15,Premio Natale di Roma (Turf),Flat,Lost President (IRE),2,7,4,1:30.27,
2,1833830,2026-04-25,919235,Firenze,16:15,Premio Natale di Roma (Turf),Flat,Heldtoransom (IRE),3,4,4,1:30.31,
3,1833831,2026-04-25,919235,Firenze,16:15,Premio Natale di Roma (Turf),Flat,Elegant Power (GB),4,3,4,1:30.68,


In [15]:
# Establish the source-wide inventory of race type values.
#
# Surface may be represented partly through the `type` field, but `type`
# may instead describe the broad racing code or discipline.
#
# For each exact raw type value, this cell measures:
# - runner rows;
# - provisional races using date + course + off;
# - distinct raw course values;
# - first and last observed dates.
#
# No type value is yet interpreted as a surface category.

type_inventory_sql = f"""
WITH provisional_race_types AS (
    SELECT
        date,
        course,
        off,
        type
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off,
        type
),
race_type_counts AS (
    SELECT
        type,
        COUNT(*) AS provisional_races
    FROM provisional_race_types
    GROUP BY type
),
runner_type_counts AS (
    SELECT
        type,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT course) AS distinct_raw_course_values,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY type
)
SELECT
    runners.type,
    runners.runner_rows,
    races.provisional_races,
    runners.distinct_raw_course_values,
    runners.first_date,
    runners.last_date
FROM runner_type_counts AS runners
JOIN race_type_counts AS races
    ON races.type = runners.type
ORDER BY
    runners.runner_rows DESC,
    runners.type
"""

type_inventory = pd.read_sql_query(type_inventory_sql, connection)

print(f"Distinct raw type values: {len(type_inventory):,}")
type_inventory

Distinct raw type values: 4


,type,runner_rows,provisional_races,distinct_raw_course_values,first_date,last_date
0,Flat,1268229,126391,475,2015-01-01,2026-05-27
1,Hurdle,358441,35462,144,2015-01-01,2026-05-27
2,Chase,179645,22547,124,2015-01-01,2026-05-27
3,NH Flat,44970,4643,92,2015-01-01,2026-05-27


In [16]:
# Inventory parenthetical markers found in race names.
#
# Surface information may appear explicitly in race_name, as observed in:
# "Premio Natale di Roma (Turf)"
#
# This cell works at provisional race grain using:
# date + course + off
#
# For each provisional race, it:
# - preserves the exact raw race_name;
# - extracts every parenthetical group;
# - counts the resulting exact marker patterns.
#
# The output is descriptive only. Parenthetical text may represent surface,
# age, sex, grade, eligibility, sponsorship, or other race conditions.

race_name_inventory_sql = f"""
SELECT
    date,
    course,
    off,
    race_name,
    type
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
GROUP BY
    date,
    course,
    off,
    race_name,
    type
"""

provisional_race_names = pd.read_sql_query(
    race_name_inventory_sql,
    connection,
)

def extract_race_name_parenthetical_pattern(race_name):
    """Return all parenthetical groups in their original order."""
    groups = re.findall(r"\([^()]+\)", str(race_name))
    return " ".join(groups) if groups else "[none]"

provisional_race_names["parenthetical_pattern"] = (
    provisional_race_names["race_name"].map(
        extract_race_name_parenthetical_pattern
    )
)

race_name_parenthetical_summary = (
    provisional_race_names
    .groupby("parenthetical_pattern", as_index=False)
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["provisional_races", "distinct_race_names", "parenthetical_pattern"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Provisional races inspected: "
    f"{len(provisional_race_names):,}"
)
print(
    "Distinct race-name parenthetical patterns: "
    f"{len(race_name_parenthetical_summary):,}"
)

race_name_parenthetical_summary.head(50)

Provisional races inspected: 189,043
Distinct race-name parenthetical patterns: 6,593


,parenthetical_pattern,provisional_races,distinct_race_names,distinct_raw_courses,first_date,last_date
0,[none],97445,50719,157,2015-01-01,2026-05-27
1,(GBB Race),10574,7342,64,2020-08-04,2026-05-27
2,(Div I),4197,2816,103,2015-01-16,2026-05-27
3,(Div II),4193,2817,103,2015-01-16,2026-05-27
4,(Plus 10 Race),3395,1893,60,2015-03-28,2021-09-02
5,(3yo+) (Turf),3170,1630,163,2015-01-01,2026-01-10
6,(3yo) (Turf),1413,693,117,2015-01-02,2026-01-18
7,(3yo+) (Course A) (Turf),1221,645,4,2017-02-02,2026-01-18
8,(3yo Fillies) (Turf),1134,451,90,2015-01-15,2025-12-26
9,(3yo+) (Course C) (Turf),959,623,4,2017-02-05,2026-03-08


In [17]:
# Measure explicit surface terminology in race names.
#
# A race name may contain several parenthetical conditions, so this cell
# searches independently for each observed surface-related term.
#
# Counts are measured at provisional race grain:
# date + course + off
#
# Categories are deliberately kept as raw evidence:
# - Turf
# - Dirt
# - Tapeta
# - Polytrack
# - All-Weather Track
# - Main Track
#
# These terms may overlap within one race name. For example,
# "All-Weather Track" and "Polytrack" can occur together.
# No single canonical surface category is assigned yet.

surface_terms = {
    "Turf": r"\(Turf\)",
    "Dirt": r"\(Dirt\)",
    "Tapeta": r"\(Tapeta\)",
    "Polytrack": r"\(Polytrack\)",
    "All-Weather Track": r"\(All-Weather Track\)",
    "Main Track": r"\(Main Track\)",
}

surface_term_rows = []

for surface_term, pattern in surface_terms.items():
    matched_races = provisional_race_names.loc[
        provisional_race_names["race_name"].str.contains(
            pattern,
            regex=True,
            na=False,
        )
    ]

    surface_term_rows.append(
        {
            "surface_term": surface_term,
            "provisional_races": len(matched_races),
            "distinct_race_names": matched_races["race_name"].nunique(),
            "distinct_raw_courses": matched_races["course"].nunique(),
            "first_date": matched_races["date"].min(),
            "last_date": matched_races["date"].max(),
        }
    )

surface_term_summary = (
    pd.DataFrame(surface_term_rows)
    .sort_values(
        ["provisional_races", "surface_term"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

# Also measure how many provisional races contain at least one of the
# recognised explicit surface terms.
combined_surface_pattern = "|".join(surface_terms.values())

races_with_explicit_surface_term = provisional_race_names.loc[
    provisional_race_names["race_name"].str.contains(
        combined_surface_pattern,
        regex=True,
        na=False,
    )
].copy()

print(
    "Provisional races with at least one recognised explicit "
    f"surface term: {len(races_with_explicit_surface_term):,}"
)
print(
    "Share of all provisional races: "
    f"{len(races_with_explicit_surface_term) / len(provisional_race_names):.2%}"
)

surface_term_summary

Provisional races with at least one recognised explicit surface term: 46,573
Share of all provisional races: 24.64%


,surface_term,provisional_races,distinct_race_names,distinct_raw_courses,first_date,last_date
0,Turf,32637,18501,344,2015-01-01,2026-05-27
1,Dirt,6869,3508,119,2015-01-01,2026-05-26
2,Polytrack,6025,3487,16,2015-01-03,2026-05-19
3,All-Weather Track,4029,1977,12,2018-08-26,2026-02-08
4,Main Track,1921,664,37,2018-08-25,2025-12-28
5,Tapeta,1012,514,8,2015-01-02,2026-05-02


In [18]:
# Test overlap and contradiction between course-based all-weather evidence
# and explicit surface terms found in race_name.
#
# Course-based evidence:
# - raw course contains "(AW)"
#
# Race-name evidence:
# - Turf
# - Dirt
# - Tapeta
# - Polytrack
# - All-Weather Track
# - Main Track
#
# The purpose is to identify:
# - confirming combinations;
# - missing race-name surface evidence;
# - potentially contradictory combinations.
#
# No final surface hierarchy is assigned yet.

surface_evidence = provisional_race_names.copy()

surface_evidence["course_marks_aw"] = (
    surface_evidence["course"].str.contains(
        r"\(AW\)",
        regex=True,
        na=False,
    )
)

for surface_term, pattern in surface_terms.items():
    column_name = (
        "race_name_marks_"
        + surface_term.lower().replace("-", "_").replace(" ", "_")
    )

    surface_evidence[column_name] = (
        surface_evidence["race_name"].str.contains(
            pattern,
            regex=True,
            na=False,
        )
    )

surface_columns = [
    column
    for column in surface_evidence.columns
    if column.startswith("race_name_marks_")
]

surface_evidence["race_name_has_explicit_surface"] = (
    surface_evidence[surface_columns].any(axis=1)
)

surface_evidence["explicit_surface_terms"] = (
    surface_evidence[surface_columns]
    .apply(
        lambda row: ", ".join(
            column.replace("race_name_marks_", "")
            for column, present in row.items()
            if present
        )
        if row.any()
        else "[none]",
        axis=1,
    )
)

course_aw_overlap_summary = (
    surface_evidence
    .groupby(
        [
            "course_marks_aw",
            "race_name_has_explicit_surface",
            "explicit_surface_terms",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["course_marks_aw", "provisional_races"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(
    "Provisional races at courses marked (AW): "
    f"{surface_evidence['course_marks_aw'].sum():,}"
)
print(
    "Of those, races with an explicit race-name surface term: "
    f"{surface_evidence.loc[
        surface_evidence['course_marks_aw'],
        'race_name_has_explicit_surface'
    ].sum():,}"
)

course_aw_overlap_summary

Provisional races at courses marked (AW): 33,023
Of those, races with an explicit race-name surface term: 715


,course_marks_aw,race_name_has_explicit_surface,explicit_surface_terms,provisional_races,distinct_raw_courses,first_date,last_date
0,True,False,[none],32308,8,2015-01-01,2026-05-27
1,True,True,tapeta,714,1,2015-01-02,2015-12-30
2,True,True,polytrack,1,1,2015-01-05,2015-01-05
3,False,False,[none],110162,202,2015-01-01,2026-05-27
4,False,True,turf,32637,344,2015-01-01,2026-05-27
5,False,True,dirt,4955,112,2015-01-01,2026-05-26
6,False,True,"polytrack, all_weather_track",3788,6,2018-08-28,2026-01-23
7,False,True,polytrack,2236,15,2015-01-03,2026-05-19
8,False,True,"dirt, main_track",1911,37,2018-08-25,2025-12-28
9,False,True,"tapeta, all_weather_track",218,6,2018-08-26,2025-12-06


In [19]:
# Identify which raw courses produce each explicit surface combination.
#
# This cell helps distinguish:
# - stable course-level conventions;
# - isolated anomalies;
# - surface terminology that changes by venue or period.
#
# It also exposes the three races labelled both Dirt and
# All-Weather Track, which may reflect a legitimate synthetic dirt track
# description rather than contradictory data.

surface_course_detail = (
    surface_evidence.loc[
        surface_evidence["race_name_has_explicit_surface"]
        | surface_evidence["course_marks_aw"]
    ]
    .groupby(
        [
            "course",
            "course_marks_aw",
            "explicit_surface_terms",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "course_marks_aw",
            "explicit_surface_terms",
            "provisional_races",
            "course",
        ],
        ascending=[False, True, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Course and surface-evidence combinations: "
    f"{len(surface_course_detail):,}"
)

surface_course_detail

Course and surface-evidence combinations: 544


,course,course_marks_aw,explicit_surface_terms,provisional_races,distinct_race_names,first_date,last_date
0,Wolverhampton (AW),True,[none],6328,2850,2015-11-10,2026-05-18
1,Kempton (AW),True,[none],5046,2159,2015-01-07,2026-05-27
2,Lingfield (AW),True,[none],4802,2230,2015-01-03,2026-05-12
3,Newcastle (AW),True,[none],4409,2301,2016-05-17,2026-05-19
4,Chelmsford (AW),True,[none],4310,2327,2015-01-11,2026-03-26
5,Southwell (AW),True,[none],3816,1660,2015-01-01,2026-05-21
6,Dundalk (AW) (IRE),True,[none],3318,1416,2015-01-02,2025-10-14
7,Dundalk (AW),True,[none],279,176,2025-10-17,2026-05-26
8,Lingfield (AW),True,polytrack,1,1,2015-01-05,2015-01-05
9,Wolverhampton (AW),True,tapeta,714,385,2015-01-02,2015-12-30


In [20]:
# Measure how many distinct explicit surface families occur at each raw course.
#
# This cell reduces the detailed race-name terminology into provisional
# surface families:
# - turf;
# - dirt;
# - synthetic_polytrack;
# - synthetic_tapeta;
# - other_all_weather;
# - no_explicit_race_name_surface.
#
# "Main Track" is treated as supporting context for Dirt rather than as
# a separate surface family.
#
# The purpose is to identify:
# - courses with one stable explicit surface family;
# - courses with multiple legitimate surfaces;
# - courses where the race name provides no explicit surface evidence.
#
# These are analytical categories only and are not final canonical rules.

def classify_explicit_surface_family(row):
    """Assign a provisional surface family from explicit race-name evidence."""
    if row["race_name_marks_turf"]:
        return "turf"

    if row["race_name_marks_dirt"]:
        return "dirt"

    if row["race_name_marks_polytrack"]:
        return "synthetic_polytrack"

    if row["race_name_marks_tapeta"]:
        return "synthetic_tapeta"

    if row["race_name_marks_all_weather_track"]:
        return "other_all_weather"

    return "no_explicit_race_name_surface"


surface_evidence["explicit_surface_family"] = (
    surface_evidence.apply(
        classify_explicit_surface_family,
        axis=1,
    )
)

course_surface_family_summary = (
    surface_evidence
    .groupby("course", as_index=False)
    .agg(
        provisional_races=("race_name", "size"),
        explicit_surface_families=(
            "explicit_surface_family",
            lambda values: ", ".join(
                sorted(
                    {
                        value
                        for value in values
                        if value != "no_explicit_race_name_surface"
                    }
                )
            )
            or "[none]",
        ),
        distinct_explicit_surface_families=(
            "explicit_surface_family",
            lambda values: len(
                {
                    value
                    for value in values
                    if value != "no_explicit_race_name_surface"
                }
            ),
        ),
        races_without_explicit_surface=(
            "race_name_has_explicit_surface",
            lambda values: (~values).sum(),
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

course_surface_family_summary["share_without_explicit_surface"] = (
    course_surface_family_summary["races_without_explicit_surface"]
    / course_surface_family_summary["provisional_races"]
)

course_surface_family_summary = (
    course_surface_family_summary
    .sort_values(
        [
            "distinct_explicit_surface_families",
            "provisional_races",
            "course",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Raw courses assessed: "
    f"{len(course_surface_family_summary):,}"
)
print(
    "Courses with more than one explicit surface family: "
    f"{(
        course_surface_family_summary[
            'distinct_explicit_surface_families'
        ] > 1
    ).sum():,}"
)
print(
    "Courses with exactly one explicit surface family: "
    f"{(
        course_surface_family_summary[
            'distinct_explicit_surface_families'
        ] == 1
    ).sum():,}"
)
print(
    "Courses with no explicit race-name surface family: "
    f"{(
        course_surface_family_summary[
            'distinct_explicit_surface_families'
        ] == 0
    ).sum():,}"
)

course_surface_family_summary.head(50)

Raw courses assessed: 528
Courses with more than one explicit surface family: 79
Courses with exactly one explicit surface family: 327
Courses with no explicit race-name surface family: 122


,course,provisional_races,explicit_surface_families,distinct_explicit_surface_families,races_without_explicit_surface,first_date,last_date,share_without_explicit_surface
0,Woodbine (CAN),434,"other_all_weather, synthetic_polytrack, synthe...",4,4,2015-04-18,2025-10-11,0.009217
1,Chantilly (FR),4140,"other_all_weather, synthetic_polytrack, turf",3,9,2015-01-23,2025-10-14,0.002174
2,Deauville (FR),4051,"other_all_weather, synthetic_polytrack, turf",3,52,2015-01-03,2025-08-30,0.012836
3,Gulfstream Park (USA),745,"dirt, synthetic_tapeta, turf",3,2,2015-01-03,2025-09-20,0.002685
4,Turfway Park (USA),24,"other_all_weather, synthetic_polytrack, synthe...",3,0,2015-02-28,2025-03-22,0.000000
5,Sha Tin (HK),4160,"dirt, turf",2,3,2015-01-25,2025-10-12,0.000721
6,Meydan (UAE),1320,"dirt, turf",2,0,2015-01-08,2025-04-05,0.000000
7,Santa Anita (USA),828,"dirt, turf",2,4,2015-01-03,2025-10-06,0.004831
8,Saratoga (USA),528,"dirt, turf",2,80,2015-07-24,2025-09-01,0.151515
9,Belmont Park (USA),482,"dirt, turf",2,87,2015-04-29,2023-07-08,0.180498


In [21]:
# Measure race-level surface coverage using only direct source evidence.
#
# Evidence hierarchy for this diagnostic:
#
# 1. An explicit surface term in race_name:
#    - Turf
#    - Dirt
#    - Polytrack
#    - Tapeta
#    - other All-Weather Track wording
#
# 2. If race_name has no explicit surface term, a course name containing
#    "(AW)" supplies broad all-weather evidence.
#
# 3. Otherwise, surface remains unresolved.
#
# This cell does not use external course knowledge or infer that all British
# Flat races are turf. It measures only what can be derived directly from the
# current source fields.

def derive_direct_surface_evidence(row):
    """Return provisional surface and evidence source from direct raw evidence."""
    if row["race_name_marks_turf"]:
        return pd.Series(
            ["turf", "race_name_explicit_turf"]
        )

    if row["race_name_marks_dirt"]:
        return pd.Series(
            ["dirt", "race_name_explicit_dirt"]
        )

    if row["race_name_marks_polytrack"]:
        return pd.Series(
            ["synthetic_polytrack", "race_name_explicit_polytrack"]
        )

    if row["race_name_marks_tapeta"]:
        return pd.Series(
            ["synthetic_tapeta", "race_name_explicit_tapeta"]
        )

    if row["race_name_marks_all_weather_track"]:
        return pd.Series(
            ["all_weather_unspecified", "race_name_explicit_all_weather"]
        )

    if row["course_marks_aw"]:
        return pd.Series(
            ["all_weather_unspecified", "course_name_aw_marker"]
        )

    return pd.Series(
        ["unresolved", "no_direct_surface_evidence"]
    )


surface_evidence[
    ["direct_surface", "surface_evidence_source"]
] = surface_evidence.apply(
    derive_direct_surface_evidence,
    axis=1,
)

direct_surface_coverage = (
    surface_evidence
    .groupby(
        ["direct_surface", "surface_evidence_source"],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

direct_surface_coverage["share_of_all_races"] = (
    direct_surface_coverage["provisional_races"]
    / len(surface_evidence)
)

direct_surface_coverage = (
    direct_surface_coverage
    .sort_values(
        ["provisional_races", "direct_surface"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

resolved_races = (
    surface_evidence["direct_surface"] != "unresolved"
).sum()

print(
    "Provisional races with directly derivable surface evidence: "
    f"{resolved_races:,}"
)
print(
    "Directly resolved share of all provisional races: "
    f"{resolved_races / len(surface_evidence):.2%}"
)
print(
    "Provisional races remaining unresolved: "
    f"{(surface_evidence['direct_surface'] == 'unresolved').sum():,}"
)

direct_surface_coverage

Provisional races with directly derivable surface evidence: 78,871
Directly resolved share of all provisional races: 41.72%
Provisional races remaining unresolved: 110,172


,direct_surface,surface_evidence_source,provisional_races,distinct_raw_courses,first_date,last_date,share_of_all_races
0,unresolved,no_direct_surface_evidence,110172,203,2015-01-01,2026-05-27,0.582788
1,turf,race_name_explicit_turf,32637,344,2015-01-01,2026-05-27,0.172643
2,all_weather_unspecified,course_name_aw_marker,32308,8,2015-01-01,2026-05-27,0.170903
3,dirt,race_name_explicit_dirt,6869,119,2015-01-01,2026-05-26,0.036336
4,synthetic_polytrack,race_name_explicit_polytrack,6025,16,2015-01-03,2026-05-19,0.031871
5,synthetic_tapeta,race_name_explicit_tapeta,1012,8,2015-01-02,2026-05-02,0.005353
6,all_weather_unspecified,race_name_explicit_all_weather,20,4,2019-08-20,2025-10-11,0.000106


In [22]:
# Profile the 110,172 races that remain unresolved after direct
# surface parsing.
#
# This cell measures unresolved races by:
# - raw race type;
# - whether the course has an explicit jurisdiction suffix;
# - whether the course is one of the longstanding unsuffixed British-course
#   values identified earlier;
# - whether it is a historically linked post-October-2025 unsuffixed
#   international course;
# - remaining unclassified course representation.
#
# The categories are diagnostic only. They do not yet assign surface.

established_unsuffixed_british_courses = set(
    unlinked_unsuffixed_courses.loc[
        unlinked_unsuffixed_courses["course"] != "Firenze",
        "course",
    ]
)

historically_linked_unsuffixed_courses = set(
    historical_jurisdiction_links["later_raw_course"]
)

def classify_unresolved_course_representation(course_name):
    """Classify how jurisdiction or venue context is represented."""
    course_name = str(course_name)

    if course_name in established_unsuffixed_british_courses:
        return "established_unsuffixed_british_course"

    if course_name in historically_linked_unsuffixed_courses:
        return "historically_linked_unsuffixed_international"

    if extract_terminal_jurisdiction_code(course_name) is not None:
        return "explicit_terminal_jurisdiction_code"

    if "(" in course_name and ")" in course_name:
        return "other_parenthetical_course_form"

    return "other_unsuffixed_or_unresolved"

unresolved_surface_races = surface_evidence.loc[
    surface_evidence["direct_surface"] == "unresolved"
].copy()

unresolved_surface_races["course_representation_group"] = (
    unresolved_surface_races["course"].map(
        classify_unresolved_course_representation
    )
)

unresolved_surface_profile = (
    unresolved_surface_races
    .groupby(
        ["course_representation_group", "type"],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

unresolved_surface_profile["share_of_unresolved_races"] = (
    unresolved_surface_profile["provisional_races"]
    / len(unresolved_surface_races)
)

unresolved_surface_profile = (
    unresolved_surface_profile
    .sort_values(
        ["provisional_races", "course_representation_group", "type"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

print(f"Unresolved provisional races profiled: {len(unresolved_surface_races):,}")
print(
    "Unresolved races at established unsuffixed British courses: "
    f"{(
        unresolved_surface_races['course_representation_group']
        == 'established_unsuffixed_british_course'
    ).sum():,}"
)
print(
    "Unresolved races at historically linked unsuffixed international courses: "
    f"{(
        unresolved_surface_races['course_representation_group']
        == 'historically_linked_unsuffixed_international'
    ).sum():,}"
)

unresolved_surface_profile

Unresolved provisional races profiled: 110,172
Unresolved races at established unsuffixed British courses: 80,770
Unresolved races at historically linked unsuffixed international courses: 1,353


,course_representation_group,type,provisional_races,distinct_raw_courses,first_date,last_date,share_of_unresolved_races
0,established_unsuffixed_british_course,Flat,39455,36,2015-02-13,2026-05-27,0.358122
1,established_unsuffixed_british_course,Hurdle,22645,43,2015-01-01,2026-05-27,0.205542
2,established_unsuffixed_british_course,Chase,15671,43,2015-01-01,2026-05-27,0.142241
3,explicit_terminal_jurisdiction_code,Flat,11457,104,2015-01-01,2025-10-13,0.103992
4,explicit_terminal_jurisdiction_code,Hurdle,9158,34,2015-01-01,2025-10-14,0.083125
5,explicit_terminal_jurisdiction_code,Chase,4572,34,2015-01-01,2025-10-12,0.041499
6,established_unsuffixed_british_course,NH Flat,2999,40,2015-01-01,2026-05-26,0.027221
7,other_parenthetical_course_form,Flat,1438,1,2015-06-19,2025-08-23,0.013052
8,explicit_terminal_jurisdiction_code,NH Flat,1424,24,2015-01-03,2025-10-14,0.012925
9,historically_linked_unsuffixed_international,Hurdle,581,20,2025-10-15,2026-05-27,0.005274


In [23]:
# Identify British venues represented by both an unsuffixed course value
# and an explicit "(AW)" course value.
#
# This tests whether the same apparent venue name is split into distinct
# surface-bearing raw course values, for example:
# - Lingfield
# - Lingfield (AW)
#
# Such pairs are important because the unsuffixed form must not inherit
# all-weather status merely from sharing a venue name.
#
# This cell compares exact raw values only. It does not yet assign turf
# to the unsuffixed form.

aw_course_values = course_inventory.loc[
    course_inventory["course"].str.contains(
        r"\(AW\)",
        regex=True,
        na=False,
    )
].copy()

aw_course_values["base_venue_name"] = (
    aw_course_values["course"]
    .str.replace(r"\s+\(AW\)", "", regex=True)
    .str.replace(r"\s+\(IRE\)$", "", regex=True)
)

unsuffixed_british_inventory = course_inventory.loc[
    course_inventory["course"].isin(
        established_unsuffixed_british_courses
    )
].copy()

dual_surface_venue_pairs = (
    aw_course_values
    .merge(
        unsuffixed_british_inventory,
        left_on="base_venue_name",
        right_on="course",
        how="inner",
        suffixes=("_aw", "_unsuffixed"),
    )
    [
        [
            "base_venue_name",
            "course_aw",
            "provisional_races_aw",
            "first_date_aw",
            "last_date_aw",
            "course_unsuffixed",
            "provisional_races_unsuffixed",
            "first_date_unsuffixed",
            "last_date_unsuffixed",
        ]
    ]
    .sort_values("base_venue_name")
    .reset_index(drop=True)
)

print(
    "Apparent British venues with both unsuffixed and (AW) forms: "
    f"{len(dual_surface_venue_pairs):,}"
)

dual_surface_venue_pairs

Apparent British venues with both unsuffixed and (AW) forms: 4


,base_venue_name,course_aw,provisional_races_aw,first_date_aw,last_date_aw,course_unsuffixed,provisional_races_unsuffixed,first_date_unsuffixed,last_date_unsuffixed
0,Kempton,Kempton (AW),5046,2015-01-07,2026-05-27,Kempton,902,2015-01-10,2026-03-23
1,Lingfield,Lingfield (AW),4803,2015-01-03,2026-05-12,Lingfield,1457,2015-01-05,2026-05-26
2,Newcastle,Newcastle (AW),4409,2016-05-17,2026-05-19,Newcastle,1010,2015-01-03,2026-04-11
3,Southwell,Southwell (AW),3816,2015-01-01,2026-05-21,Southwell,1372,2015-02-22,2026-05-11


In [24]:
# Test the unresolved unsuffixed races at the four dual-representation venues.
#
# The source distinguishes:
# - venue name alone;
# - venue name with "(AW)".
#
# This cell profiles only the unsuffixed form by race type and date range.
# The purpose is to see whether those records behave like a separate
# non-all-weather course configuration rather than incomplete copies of
# the "(AW)" form.
#
# No turf assignment is made yet.

dual_representation_venues = set(
    dual_surface_venue_pairs["base_venue_name"]
)

dual_venue_unsuffixed_profile = (
    unresolved_surface_races.loc[
        unresolved_surface_races["course"].isin(
            dual_representation_venues
        )
    ]
    .groupby(
        ["course", "type"],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        active_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["course", "provisional_races"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print(
    "Unresolved races at unsuffixed dual-representation venues: "
    f"{dual_venue_unsuffixed_profile['provisional_races'].sum():,}"
)

dual_venue_unsuffixed_profile

Unresolved races at unsuffixed dual-representation venues: 4,738


,course,type,provisional_races,distinct_race_names,active_dates,first_date,last_date
0,Kempton,Hurdle,482,362,130,2015-01-10,2026-03-23
1,Kempton,Chase,362,265,130,2015-01-10,2026-03-23
2,Kempton,NH Flat,58,51,37,2015-02-06,2026-03-14
3,Lingfield,Flat,992,775,172,2015-05-08,2026-05-26
4,Lingfield,Hurdle,265,220,76,2015-01-05,2026-02-05
5,Lingfield,Chase,197,173,76,2015-01-05,2026-02-05
6,Lingfield,NH Flat,3,3,3,2023-11-14,2026-01-19
7,Newcastle,Hurdle,450,393,122,2015-01-03,2026-04-11
8,Newcastle,Chase,318,276,123,2015-01-03,2026-04-11
9,Newcastle,NH Flat,121,110,83,2015-01-03,2026-04-11


In [25]:
# Inspect all unsuffixed Flat races at the four venues that also have
# a separate "(AW)" course value.
#
# This isolates:
# - Lingfield turf Flat racing;
# - Newcastle's pre-all-weather Flat history;
# - the single Southwell Flat exception;
# - confirmation that Kempton has no unsuffixed Flat records.
#
# The output remains at provisional race grain and preserves the exact
# raw course, race name and source attributes.

dual_venue_unsuffixed_flat_sql = f"""
SELECT
    date,
    course,
    off,
    race_name,
    type,
    race_id,
    COUNT(*) AS runner_rows,
    MIN(rowid) AS first_source_rowid,
    MAX(rowid) AS last_source_rowid
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND course IN ('Kempton', 'Lingfield', 'Newcastle', 'Southwell')
  AND type = 'Flat'
GROUP BY
    date,
    course,
    off,
    race_name,
    type,
    race_id
ORDER BY
    course,
    date,
    off
"""

dual_venue_unsuffixed_flat_races = pd.read_sql_query(
    dual_venue_unsuffixed_flat_sql,
    connection,
)

print(
    "Unsuffixed Flat provisional races at the four dual-representation venues: "
    f"{len(dual_venue_unsuffixed_flat_races):,}"
)

print()
print("Counts by course:")
print(
    dual_venue_unsuffixed_flat_races["course"]
    .value_counts()
    .sort_index()
    .to_string()
)

dual_venue_unsuffixed_flat_races.loc[
    dual_venue_unsuffixed_flat_races["course"].isin(
        ["Newcastle", "Southwell"]
    )
]

Unsuffixed Flat provisional races at the four dual-representation venues: 1,114

Counts by course:
course
Lingfield    992
Newcastle    121
Southwell      1


,date,course,off,race_name,type,race_id,runner_rows,first_source_rowid,last_source_rowid
992,2015-04-11,Newcastle,1:40,vertem.co.uk Maiden Fillies Stakes,Flat,621254,9,37296,37304
993,2015-04-11,Newcastle,2:15,Protecting Your Wealth Maiden Stakes,Flat,621255,8,37287,37310
994,2015-04-11,Newcastle,2:45,@VertemAM Twitter Handicap,Flat,621256,16,37264,37285
995,2015-04-11,Newcastle,3:15,Perfect Partnerships Handicap,Flat,621257,13,36744,37283
996,2015-04-11,Newcastle,3:55,Vertem Management Handicap (Div I),Flat,621258,11,36749,36759
997,2015-04-11,Newcastle,5:00,Vertem Management Handicap (Div II),Flat,623929,11,36737,36797
998,2015-04-11,Newcastle,5:35,Fresh Approach At Vertem Handicap,Flat,621259,12,36720,36764
999,2015-04-11,Newcastle,6:05,Investing For The Future Apprentice Handicap,Flat,621260,15,36718,36740
1000,2015-04-28,Newcastle,5:10,EBF Stallions/Hardwick Hall Hotel Novice Stakes,Flat,622992,4,45655,45817
1001,2015-04-28,Newcastle,5:45,Bowburn Hall Hotel Maiden Stakes,Flat,622993,8,45587,45595


In [26]:
# Inspect the ambiguous unsuffixed Newcastle and exceptional Southwell Flat
# records at runner grain.
#
# Newcastle:
# - pre-October-2025 unsuffixed Flat records may be historic British turf;
# - post-October-2025 unsuffixed Flat records may represent Newcastle (AUS)
#   after jurisdiction suffixes were removed.
#
# Southwell:
# - one unsuffixed Flat race appears in 2023 despite the separate
#   "Southwell (AW)" representation.
#
# This cell displays fields that may provide jurisdiction, surface, meeting,
# or transferred-race evidence. No correction is applied.

ambiguous_flat_rows_sql = f"""
SELECT
    rowid AS source_rowid,
    date,
    race_id,
    course,
    off,
    race_name,
    type,
    ran,
    horse,
    jockey,
    trainer,
    draw,
    pos,
    time,
    comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND type = 'Flat'
  AND (
        (course = 'Newcastle' AND date >= '2025-10-15')
        OR
        (course = 'Southwell')
      )
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

ambiguous_flat_rows = pd.read_sql_query(
    ambiguous_flat_rows_sql,
    connection,
)

print(f"Runner rows inspected: {len(ambiguous_flat_rows):,}")
print()
print("Provisional races:")
display(
    ambiguous_flat_rows[
        ["date", "course", "off", "race_name", "race_id", "ran"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

ambiguous_flat_rows

Runner rows inspected: 43

Provisional races:


,date,course,off,race_name,race_id,ran
0,2023-06-26,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,842254,6
1,2025-11-15,Newcastle,05:05,NZB 3yo Spring Stakes (Turf),908281,11
2,2025-11-15,Newcastle,05:45,The Newcastle Herald Hunter (Handicap) (3yo+) ...,908282,16
3,2026-03-06,Newcastle,05:55,Horsepower Newcastle Stakes (Handicap) (Turf),914933,10


,source_rowid,date,race_id,course,off,race_name,type,ran,horse,jockey,trainer,draw,pos,time,comment
0,1361686,2023-06-26,842254,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,6,Jenner Park (IRE),Miss Eleanor Williams,Evan Williams,,6,3:57.17,Took keen hold - raced wide early - led - head...
1,1361687,2023-06-26,842254,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,6,Upbeat Betty (GB),Isabel Williams,Evan Williams,,5,3:56.17,Towards rear - headway on outer 5f out - weake...
2,1361688,2023-06-26,842254,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,6,Happy Friend (IRE),Tom Cannon,Karen Jewell,,4,3:55.77,Prominent - outpaced 4f out - dropped to rear ...
3,1361689,2023-06-26,842254,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,6,Gladys Eva Jane (GB),Sean Bowen,Peter Bowen,,3,3:54.69,In touch with leaders - headway and pressed le...
4,1361690,2023-06-26,842254,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,6,Riccirella (GB),Charlie Todd,Michael Dods,,2,3:54.67,Held up in rear - going easily but waiting for...
5,1361691,2023-06-26,842254,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,6,Cast A Spell (IRE),Jamie Moore,Gary Moore,,1,3:54.32,Raced in second - led 5f out - pushed along 3f...
6,1766003,2025-11-15,908281,Newcastle,05:05,NZB 3yo Spring Stakes (Turf),Flat,11,Green Spaces (AUS),Rachel King,Bjorn Baker,1,1,1:36.00,Dwelt at start - settled in midfield on inner ...
7,1766004,2025-11-15,908281,Newcastle,05:05,NZB 3yo Spring Stakes (Turf),Flat,11,Bird Whistle (AUS),Braith Nock,Brett & Georgie Cavanough,4,2,1:36.50,
8,1766005,2025-11-15,908281,Newcastle,05:05,NZB 3yo Spring Stakes (Turf),Flat,11,Long Legs (AUS),Tyler Schiller,Gary Portelli,10,3,1:36.58,
9,1766006,2025-11-15,908281,Newcastle,05:05,NZB 3yo Spring Stakes (Turf),Flat,11,Grand Prairie (AUS),Thomas Sherry,Peter Snowden,2,4,1:36.88,


In [27]:
# Compare the exceptional Southwell Flat race with other races from the same
# meeting and with the same race name elsewhere in the source.
#
# The aim is to determine whether:
# - the meeting contains normal jumps or NH Flat races;
# - the same named race appears elsewhere with a different raw type;
# - the exceptional record is likely a source classification anomaly.
#
# This cell preserves all raw source values and does not correct the record.

southwell_exception_context_sql = f"""
WITH target_race AS (
    SELECT DISTINCT
        date,
        course,
        off,
        race_name
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND date = '2023-06-26'
      AND course = 'Southwell'
      AND off = '3:00'
),
same_meeting AS (
    SELECT
        'same_meeting' AS comparison_group,
        date,
        course,
        off,
        race_name,
        type,
        race_id,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT draw) AS distinct_draw_values,
        MIN(time) AS minimum_raw_time,
        MAX(time) AS maximum_raw_time
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND date = '2023-06-26'
      AND course = 'Southwell'
    GROUP BY
        date,
        course,
        off,
        race_name,
        type,
        race_id
),
same_race_name AS (
    SELECT
        'same_race_name' AS comparison_group,
        date,
        course,
        off,
        race_name,
        type,
        race_id,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT draw) AS distinct_draw_values,
        MIN(time) AS minimum_raw_time,
        MAX(time) AS maximum_raw_time
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND race_name = (
          SELECT race_name
          FROM target_race
          LIMIT 1
      )
    GROUP BY
        date,
        course,
        off,
        race_name,
        type,
        race_id
)
SELECT *
FROM same_meeting

UNION ALL

SELECT *
FROM same_race_name

ORDER BY
    comparison_group,
    date,
    course,
    off
"""

southwell_exception_context = pd.read_sql_query(
    southwell_exception_context_sql,
    connection,
)

print(
    "Context rows returned: "
    f"{len(southwell_exception_context):,}"
)

southwell_exception_context

Context rows returned: 8


,comparison_group,date,course,off,race_name,type,race_id,runner_rows,distinct_draw_values,minimum_raw_time,maximum_raw_time
0,same_meeting,2023-06-26,Southwell,1:55,Prestige Safety First Aid Training Handicap Chase,Chase,842248,6,1,-,5:9.25
1,same_meeting,2023-06-26,Southwell,2:25,Prestige Safety Your Trusted Safety Partner Ha...,Chase,842253,5,1,-,4:3.07
2,same_meeting,2023-06-26,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,842254,6,1,3:54.32,3:57.17
3,same_meeting,2023-06-26,Southwell,3:30,Prestige Safety Fire Risk Assessments Novices ...,Hurdle,842252,6,1,3:50.46,3:57.36
4,same_meeting,2023-06-26,Southwell,4:00,Prestige Safety Construction Safety Specialist...,Hurdle,842251,5,1,-,3:59.97
5,same_meeting,2023-06-26,Southwell,4:30,Prestige Safety Your Competent H&S Specialist ...,Hurdle,842249,7,1,-,6:7.76
6,same_meeting,2023-06-26,Southwell,5:00,Prestige Safety Services PPE And Workwear Cond...,Hurdle,842250,9,1,-,5:7.65
7,same_race_name,2023-06-26,Southwell,3:00,Prestige Safety e-learning @ prestigesafetyser...,Flat,842254,6,1,3:54.32,3:57.17


In [28]:
# Inspect the full descriptive fields for the exceptional Southwell race.
#
# The notebook display truncated the race name, and the meeting-level evidence
# suggests this may be a National Hunt Flat race recorded as type = "Flat".
#
# This cell retrieves one representative source row and preserves the exact
# raw values for fields that may clarify the race discipline:
# - full race name;
# - distance;
# - age and class conditions;
# - going;
# - weight and draw representation.
#
# No source value is corrected.

southwell_exception_description_sql = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_name,
    type,
    dist,
    age,
    class,
    going,
    ran,
    draw,
    wgt,
    horse,
    jockey,
    trainer,
    time
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND date = '2023-06-26'
  AND course = 'Southwell'
  AND off = '3:00'
ORDER BY rowid
LIMIT 1
"""

southwell_exception_description = pd.read_sql_query(
    southwell_exception_description_sql,
    connection,
)

record = southwell_exception_description.iloc[0]

for field, value in record.items():
    print(f"{field}: {value}")

source_rowid: 1361686
date: 2023-06-26
course: Southwell
off: 3:00
race_name: Prestige Safety e-learning @ prestigesafetyservices.com Mares Open NHF Race (Cat 1 Elim) (GBB)
type: Flat
dist: 2m
age: 4
class: Class 5
going: Good
ran: 6
draw: 
wgt: 10-4
horse: Jenner Park (IRE)
jockey: Miss Eleanor Williams
trainer: Evan Williams
time: 3:57.17


In [29]:
# Test source-wide consistency between explicit NH Flat wording in race_name
# and the raw type field.
#
# The Southwell exception contains "NHF Race" in race_name but type = "Flat".
# This cell searches provisional races for common explicit National Hunt Flat
# wording and compares it with the recorded type.
#
# Search terms are deliberately narrow:
# - NHF
# - National Hunt Flat
# - Bumper
#
# This is anomaly detection, not a complete NH Flat classifier. Some genuine
# NH Flat races may not contain any of these terms.

nh_flat_wording_pattern = (
    r"\bNHF\b"
    r"|National Hunt Flat"
    r"|\bBumper\b"
)

explicit_nh_flat_races = provisional_race_names.loc[
    provisional_race_names["race_name"].str.contains(
        nh_flat_wording_pattern,
        case=False,
        regex=True,
        na=False,
    )
].copy()

explicit_nh_flat_type_summary = (
    explicit_nh_flat_races
    .groupby("type", as_index=False)
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["provisional_races", "type"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

explicit_nh_flat_mismatches = (
    explicit_nh_flat_races.loc[
        explicit_nh_flat_races["type"] != "NH Flat"
    ]
    .sort_values(
        ["date", "course", "off", "race_name"]
    )
    .reset_index(drop=True)
)

print(
    "Provisional races with explicit NH Flat wording: "
    f"{len(explicit_nh_flat_races):,}"
)
print(
    "Those not recorded as type = 'NH Flat': "
    f"{len(explicit_nh_flat_mismatches):,}"
)

display(explicit_nh_flat_type_summary)
explicit_nh_flat_mismatches

Provisional races with explicit NH Flat wording: 3,036
Those not recorded as type = 'NH Flat': 17


,type,provisional_races,distinct_raw_courses,distinct_race_names,first_date,last_date
0,NH Flat,3019,53,2492,2015-01-01,2026-05-26
1,Flat,14,4,7,2016-04-26,2025-04-29
2,Chase,2,2,2,2022-04-03,2025-11-11
3,Hurdle,1,1,1,2024-02-24,2024-02-24


,date,course,off,race_name,type,parenthetical_pattern
0,2016-04-26,Punchestown (IRE),6:05,Goffs Land Rover Bumper INH Flat,Flat,[none]
1,2017-04-25,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,[none]
2,2018-04-24,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,[none]
3,2018-06-08,Stratford,9:00,Irish Thoroughbred Marketing Champion Point-To...,Flat,(A Standard NHF Race) (Amateur Riders)
4,2019-04-30,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,[none]
5,2019-05-31,Stratford,8:50,Irish Thoroughbred Marketing Champion Point-To...,Flat,(A Standard NHF Race) (Amateur Riders)
6,2021-04-27,Punchestown (IRE),5:55,Goffs Land Rover Bumper,Flat,[none]
7,2021-05-28,Stratford,8:40,Irish Thoroughbred Marketing Champion Point-To...,Flat,(Standard NHF Race) (GBB Race)
8,2022-04-03,Fairyhouse (IRE),2:55,Book Your Bumper Bundle For Easter Handicap Chase,Chase,[none]
9,2022-04-26,Punchestown (IRE),6:00,Goffs Land Rover Bumper,Flat,[none]


In [30]:
# Inspect the 17 explicit-NH-Flat wording mismatches with fuller race context.
#
# This cell retrieves one representative row per provisional race and includes:
# - the complete race name;
# - recorded type;
# - distance, class, age and going;
# - draw availability;
# - winning time;
# - runner count.
#
# The purpose is to separate:
# - genuine discipline misclassifications;
# - source conventions where Irish bumpers are recorded as Flat;
# - keyword false positives such as "Bumper Bundle".
#
# No correction is applied.

nh_flat_mismatch_context_sql = f"""
WITH mismatch_races AS (
    SELECT
        date,
        course,
        off,
        race_name,
        type
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND (
            LOWER(race_name) LIKE '%nhf%'
            OR LOWER(race_name) LIKE '%national hunt flat%'
            OR LOWER(race_name) LIKE '%bumper%'
          )
      AND type <> 'NH Flat'
    GROUP BY
        date,
        course,
        off,
        race_name,
        type
),
ranked_rows AS (
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.off,
        d.race_name,
        d.type,
        d.dist,
        d.age,
        d.class,
        d.going,
        d.draw,
        d.ran,
        d.time,
        d.horse,
        ROW_NUMBER() OVER (
            PARTITION BY
                d.date,
                d.course,
                d.off,
                d.race_name,
                d.type
            ORDER BY
                CASE
                    WHEN d.pos = '1' THEN 0
                    ELSE 1
                END,
                d.rowid
        ) AS row_rank
    FROM {SOURCE_TABLE} AS d
    JOIN mismatch_races AS m
        ON m.date = d.date
       AND m.course = d.course
       AND m.off = d.off
       AND m.race_name = d.race_name
       AND m.type = d.type
    WHERE {DATA_ROW_PREDICATE}
)
SELECT
    source_rowid,
    date,
    course,
    off,
    race_name,
    type,
    dist,
    age,
    class,
    going,
    draw,
    ran,
    time,
    horse
FROM ranked_rows
WHERE row_rank = 1
ORDER BY
    date,
    course,
    off
"""

nh_flat_mismatch_context = pd.read_sql_query(
    nh_flat_mismatch_context_sql,
    connection,
)

print(
    "Mismatch races inspected: "
    f"{len(nh_flat_mismatch_context):,}"
)

pd.set_option("display.max_colwidth", None)
nh_flat_mismatch_context

Mismatch races inspected: 18


,source_rowid,date,course,off,race_name,type,dist,age,class,going,draw,ran,time,horse
0,197719,2016-04-26,Punchestown (IRE),6:05,Goffs Land Rover Bumper INH Flat,Flat,2m,4,,Good To Yielding,,16,4:2.90,Coeur De Lion (FR)
1,361159,2017-04-25,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,2m,4,,Good To Yielding,,16,3:54.50,Vision Des Flos (FR)
2,525523,2018-04-24,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,2m½f,4,,Yielding To Soft,,16,4:18.40,Commander Of Fleet (IRE)
3,551769,2018-06-08,Stratford,9:00,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (A Standard NHF Race) (Amateur Riders),Flat,2m½f,4,Class 4,Good,,9,3:56.60,Lily The Pink (GB)
4,704862,2019-04-30,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,2m½f,4,,Yielding,,19,4:3.40,Festival Dex (FR)
5,722825,2019-05-31,Stratford,8:50,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (A Standard NHF Race) (Amateur Riders),Flat,2m½f,6,Class 4,Good,,13,4:1.80,Love Around (GB)
6,1000892,2021-04-27,Punchestown (IRE),5:55,Goffs Land Rover Bumper,Flat,2m½f,4,,Yielding,,22,4:6.00,Adamantly Chosen (IRE)
7,1017876,2021-05-28,Stratford,8:40,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (Standard NHF Race) (GBB Race),Flat,2m½f,5,Class 4,Good,,5,4:14.20,Fountains Chief (GB)
8,1162684,2022-04-03,Fairyhouse (IRE),2:55,Book Your Bumper Bundle For Easter Handicap Chase,Chase,2m5½f,7,,Yielding,,13,5:35.90,Song Of Earth (IRE)
9,1174283,2022-04-26,Punchestown (IRE),6:00,Goffs Land Rover Bumper,Flat,2m½f,4,,Good To Yielding,,15,4:2.00,Absolute Notions (IRE)


In [31]:
# Create a narrower explicit NH Flat anomaly test.
#
# The previous search produced false positives where "Bumper" appeared only
# in promotional wording such as "Bumper Bundle".
#
# This stronger diagnostic accepts:
# - NHF;
# - NHFlat;
# - National Hunt Flat;
# - INH Flat;
# - race names ending in or explicitly describing a Bumper;
#
# It excludes the known promotional phrase "Bumper Bundle".
#
# The result remains a candidate anomaly set. It does not overwrite raw type.

strong_nh_flat_pattern = (
    r"\bNHF\b"
    r"|\bNHFlat\b"
    r"|National Hunt Flat"
    r"|\bINH Flat\b"
    r"|\bBumper\b"
)

strong_explicit_nh_flat_races = provisional_race_names.loc[
    provisional_race_names["race_name"].str.contains(
        strong_nh_flat_pattern,
        case=False,
        regex=True,
        na=False,
    )
    &
    ~provisional_race_names["race_name"].str.contains(
        r"Bumper Bundle",
        case=False,
        regex=True,
        na=False,
    )
].copy()

strong_nh_flat_type_summary = (
    strong_explicit_nh_flat_races
    .groupby("type", as_index=False)
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        ["provisional_races", "type"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

strong_nh_flat_mismatches = (
    strong_explicit_nh_flat_races.loc[
        strong_explicit_nh_flat_races["type"] != "NH Flat"
    ]
    .sort_values(
        ["date", "course", "off", "race_name"]
    )
    .reset_index(drop=True)
)

print(
    "Provisional races with strong explicit NH Flat wording: "
    f"{len(strong_explicit_nh_flat_races):,}"
)
print(
    "Strong explicit NH Flat races not recorded as type = 'NH Flat': "
    f"{len(strong_nh_flat_mismatches):,}"
)

display(strong_nh_flat_type_summary)
strong_nh_flat_mismatches

Provisional races with strong explicit NH Flat wording: 4,437
Strong explicit NH Flat races not recorded as type = 'NH Flat': 16


,type,provisional_races,distinct_raw_courses,distinct_race_names,first_date,last_date
0,NH Flat,4421,92,3287,2015-01-01,2026-05-27
1,Flat,15,5,8,2016-04-26,2025-04-29
2,Chase,1,1,1,2025-11-11,2025-11-11


,date,course,off,race_name,type,parenthetical_pattern
0,2016-04-26,Punchestown (IRE),6:05,Goffs Land Rover Bumper INH Flat,Flat,[none]
1,2017-04-25,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,[none]
2,2018-04-24,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,[none]
3,2018-06-08,Stratford,9:00,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (A Standard NHF Race) (Amateur Riders),Flat,(A Standard NHF Race) (Amateur Riders)
4,2019-04-30,Punchestown (IRE),6:05,Goffs Land Rover Bumper,Flat,[none]
5,2019-05-31,Stratford,8:50,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (A Standard NHF Race) (Amateur Riders),Flat,(A Standard NHF Race) (Amateur Riders)
6,2021-04-27,Punchestown (IRE),5:55,Goffs Land Rover Bumper,Flat,[none]
7,2021-05-28,Stratford,8:40,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (Standard NHF Race) (GBB Race),Flat,(Standard NHF Race) (GBB Race)
8,2022-04-26,Punchestown (IRE),6:00,Goffs Land Rover Bumper,Flat,[none]
9,2022-12-21,Lingfield (AW),12:30,Download The At The Races App Mares Open Maiden NHFlat Race (Category 3 Elimination) (GBB Race),Flat,(Category 3 Elimination) (GBB Race)


In [32]:
# Measure provisional race-level jurisdiction coverage using only the
# evidence established in this notebook.
#
# Evidence sources:
#
# 1. Recognised terminal jurisdiction code in raw course:
#    e.g. "(IRE)", "(FR)", "(AUS)".
#
# 2. Historical suffix linkage for later unsuffixed international forms:
#    e.g. "Chantilly" linked to earlier "Chantilly (FR)".
#
# 3. The established longstanding unsuffixed British-course inventory.
#
# Important exceptions:
# - Ascot, Sandown and Newcastle have cross-jurisdiction name collisions.
# - Unsuffixed Newcastle changes meaning by race context and date.
# - Firenze has no direct jurisdiction evidence.
#
# For this diagnostic, ambiguous collision names remain unresolved rather
# than being forced into one jurisdiction.

terminal_code_to_jurisdiction = {
    "ARG": "Argentina",
    "AUS": "Australia",
    "BEL": "Belgium",
    "BHR": "Bahrain",
    "BRZ": "Brazil",
    "CAN": "Canada",
    "CHI": "Chile",
    "CHN": "China",
    "CZE": "Czech Republic",
    "DEN": "Denmark",
    "FR": "France",
    "GER": "Germany",
    "GUE": "Guernsey",
    "HK": "Hong Kong",
    "HUN": "Hungary",
    "IRE": "Ireland",
    "ITY": "Italy",
    "JER": "Jersey",
    "JPN": "Japan",
    "KOR": "South Korea",
    "KSA": "Saudi Arabia",
    "NOR": "Norway",
    "NZ": "New Zealand",
    "PER": "Peru",
    "POL": "Poland",
    "QA": "Qatar",
    "SAF": "South Africa",
    "SIN": "Singapore",
    "SPA": "Spain",
    "SWE": "Sweden",
    "SWI": "Switzerland",
    "TUR": "Turkey",
    "UAE": "United Arab Emirates",
    "URU": "Uruguay",
    "USA": "United States",
}

historical_course_to_code = dict(
    zip(
        historical_jurisdiction_links["later_raw_course"],
        historical_jurisdiction_links["historical_jurisdiction_code"],
    )
)

known_cross_jurisdiction_collision_names = {
    "Ascot",
    "Sandown",
    "Newcastle",
}

def derive_provisional_jurisdiction(course_name):
    """
    Derive jurisdiction and evidence source conservatively.

    Ambiguous unsuffixed collision names are deliberately left unresolved.
    """
    course_name = str(course_name)

    terminal_code = extract_terminal_jurisdiction_code(course_name)

    if terminal_code is not None:
        return pd.Series(
            [
                terminal_code_to_jurisdiction[terminal_code],
                "explicit_terminal_course_code",
            ]
        )

    if course_name in known_cross_jurisdiction_collision_names:
        return pd.Series(
            [
                "unresolved",
                "cross_jurisdiction_course_name_collision",
            ]
        )

    if course_name in historical_course_to_code:
        historical_code = historical_course_to_code[course_name]

        return pd.Series(
            [
                terminal_code_to_jurisdiction[historical_code],
                "historical_suffixed_course_link",
            ]
        )

    if course_name in established_unsuffixed_british_courses:
        return pd.Series(
            [
                "Great Britain",
                "established_unsuffixed_british_course",
            ]
        )

    return pd.Series(
        [
            "unresolved",
            "no_established_jurisdiction_evidence",
        ]
    )


jurisdiction_evidence = provisional_race_names.copy()

jurisdiction_evidence[
    ["provisional_jurisdiction", "jurisdiction_evidence_source"]
] = jurisdiction_evidence["course"].apply(
    derive_provisional_jurisdiction
)

jurisdiction_coverage_summary = (
    jurisdiction_evidence
    .groupby(
        [
            "provisional_jurisdiction",
            "jurisdiction_evidence_source",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

jurisdiction_coverage_summary["share_of_all_races"] = (
    jurisdiction_coverage_summary["provisional_races"]
    / len(jurisdiction_evidence)
)

jurisdiction_coverage_summary = (
    jurisdiction_coverage_summary
    .sort_values(
        ["provisional_races", "provisional_jurisdiction"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

resolved_jurisdiction_races = (
    jurisdiction_evidence["provisional_jurisdiction"] != "unresolved"
).sum()

print(
    "Provisional races with jurisdiction resolved from established evidence: "
    f"{resolved_jurisdiction_races:,}"
)
print(
    "Resolved share of all provisional races: "
    f"{resolved_jurisdiction_races / len(jurisdiction_evidence):.2%}"
)
print(
    "Provisional races remaining unresolved: "
    f"{(
        jurisdiction_evidence['provisional_jurisdiction']
        == 'unresolved'
    ).sum():,}"
)

jurisdiction_coverage_summary

Provisional races with jurisdiction resolved from established evidence: 153,644
Resolved share of all provisional races: 81.27%
Provisional races remaining unresolved: 35,399


,provisional_jurisdiction,jurisdiction_evidence_source,provisional_races,distinct_raw_courses,first_date,last_date,share_of_all_races
0,Great Britain,established_unsuffixed_british_course,76260,55,2015-01-01,2026-05-27,0.403400
1,unresolved,no_established_jurisdiction_evidence,30865,8,2015-01-01,2026-05-27,0.163270
2,Ireland,explicit_terminal_course_code,29215,27,2015-01-01,2025-10-14,0.154542
3,France,explicit_terminal_course_code,19361,73,2015-01-03,2025-10-14,0.102416
4,Hong Kong,explicit_terminal_course_code,6847,2,2015-01-25,2025-10-12,0.036219
5,United States,explicit_terminal_course_code,5863,56,2015-01-01,2025-10-12,0.031014
6,unresolved,cross_jurisdiction_course_name_collision,4534,3,2015-01-03,2026-05-09,0.023984
7,Australia,explicit_terminal_course_code,3798,51,2015-01-01,2025-10-11,0.020091
8,United Arab Emirates,explicit_terminal_course_code,2219,5,2015-01-04,2025-04-12,0.011738
9,Ireland,historical_suffixed_course_link,1568,23,2025-10-15,2026-05-27,0.008294


In [33]:
# Inspect every raw course value still unresolved by the provisional
# jurisdiction derivation.
#
# The summary shows:
# - 30,865 unresolved races across eight course values with no established
#   jurisdiction evidence;
# - 4,534 races across the three known cross-jurisdiction collision names.
#
# This cell lists the exact values, race types, frequencies and date ranges
# before we add any further jurisdiction rules.
#
# No jurisdiction is assigned or corrected here.

unresolved_jurisdiction_courses = (
    jurisdiction_evidence.loc[
        jurisdiction_evidence["provisional_jurisdiction"] == "unresolved"
    ]
    .groupby(
        [
            "jurisdiction_evidence_source",
            "course",
            "type",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        active_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "jurisdiction_evidence_source",
            "course",
            "provisional_races",
            "type",
        ],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)

print(
    "Unresolved raw course values: "
    f"{unresolved_jurisdiction_courses['course'].nunique():,}"
)
print(
    "Unresolved provisional races: "
    f"{unresolved_jurisdiction_courses['provisional_races'].sum():,}"
)

unresolved_jurisdiction_courses

Unresolved raw course values: 11
Unresolved provisional races: 35,399


,jurisdiction_evidence_source,course,type,provisional_races,distinct_race_names,active_dates,first_date,last_date
0,cross_jurisdiction_course_name_collision,Ascot,Flat,1265,665,202,2015-04-29,2026-05-09
1,cross_jurisdiction_course_name_collision,Ascot,Hurdle,276,205,85,2015-01-17,2026-03-29
2,cross_jurisdiction_course_name_collision,Ascot,Chase,252,167,85,2015-01-17,2026-03-29
3,cross_jurisdiction_course_name_collision,Ascot,NH Flat,44,32,44,2015-02-14,2026-02-14
4,cross_jurisdiction_course_name_collision,Newcastle,Hurdle,450,393,122,2015-01-03,2026-04-11
5,cross_jurisdiction_course_name_collision,Newcastle,Chase,318,276,123,2015-01-03,2026-04-11
6,cross_jurisdiction_course_name_collision,Newcastle,Flat,121,116,18,2015-04-11,2026-03-06
7,cross_jurisdiction_course_name_collision,Newcastle,NH Flat,121,110,83,2015-01-03,2026-04-11
8,cross_jurisdiction_course_name_collision,Sandown,Flat,1050,723,161,2015-02-13,2026-04-24
9,cross_jurisdiction_course_name_collision,Sandown,Hurdle,328,227,94,2015-01-03,2026-04-25


In [34]:
# Extend the provisional jurisdiction derivation with explicitly curated
# British course configurations established in this notebook.
#
# These values were missed by the earlier rule because their parentheses
# describe surface or course configuration rather than jurisdiction:
#
# - seven British all-weather course forms;
# - Newmarket's July Course.
#
# Firenze and the three cross-jurisdiction collision names remain unresolved.
# Raw course values are preserved unchanged.

curated_british_course_configurations = {
    "Chelmsford (AW)",
    "Kempton (AW)",
    "Lingfield (AW)",
    "Newcastle (AW)",
    "Newmarket (July)",
    "Southwell (AW)",
    "Wolverhampton (AW)",
}

def derive_refined_provisional_jurisdiction(course_name):
    """
    Derive jurisdiction conservatively using established and curated evidence.
    """
    course_name = str(course_name)

    terminal_code = extract_terminal_jurisdiction_code(course_name)

    if terminal_code is not None:
        return pd.Series(
            [
                terminal_code_to_jurisdiction[terminal_code],
                "explicit_terminal_course_code",
            ]
        )

    if course_name in known_cross_jurisdiction_collision_names:
        return pd.Series(
            [
                "unresolved",
                "cross_jurisdiction_course_name_collision",
            ]
        )

    if course_name in historical_course_to_code:
        historical_code = historical_course_to_code[course_name]

        return pd.Series(
            [
                terminal_code_to_jurisdiction[historical_code],
                "historical_suffixed_course_link",
            ]
        )

    if course_name in curated_british_course_configurations:
        return pd.Series(
            [
                "Great Britain",
                "curated_british_course_configuration",
            ]
        )

    if course_name in established_unsuffixed_british_courses:
        return pd.Series(
            [
                "Great Britain",
                "established_unsuffixed_british_course",
            ]
        )

    return pd.Series(
        [
            "unresolved",
            "no_established_jurisdiction_evidence",
        ]
    )


refined_jurisdiction_evidence = provisional_race_names.copy()

refined_jurisdiction_evidence[
    ["provisional_jurisdiction", "jurisdiction_evidence_source"]
] = refined_jurisdiction_evidence["course"].apply(
    derive_refined_provisional_jurisdiction
)

refined_jurisdiction_summary = (
    refined_jurisdiction_evidence
    .groupby(
        [
            "provisional_jurisdiction",
            "jurisdiction_evidence_source",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

refined_jurisdiction_summary["share_of_all_races"] = (
    refined_jurisdiction_summary["provisional_races"]
    / len(refined_jurisdiction_evidence)
)

refined_jurisdiction_summary = (
    refined_jurisdiction_summary
    .sort_values(
        ["provisional_races", "provisional_jurisdiction"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

remaining_unresolved = refined_jurisdiction_evidence.loc[
    refined_jurisdiction_evidence["provisional_jurisdiction"]
    == "unresolved"
]

print(
    "Provisional races with jurisdiction resolved: "
    f"{len(refined_jurisdiction_evidence) - len(remaining_unresolved):,}"
)
print(
    "Resolved share of all provisional races: "
    f"{1 - len(remaining_unresolved) / len(refined_jurisdiction_evidence):.2%}"
)
print(
    "Provisional races remaining unresolved: "
    f"{len(remaining_unresolved):,}"
)
print()
print("Remaining unresolved raw course values:")
print(
    remaining_unresolved["course"]
    .value_counts()
    .sort_index()
    .to_string()
)

refined_jurisdiction_summary

Provisional races with jurisdiction resolved: 184,508
Resolved share of all provisional races: 97.60%
Provisional races remaining unresolved: 4,535

Remaining unresolved raw course values:
course
Ascot        1837
Firenze         1
Newcastle    1010
Sandown      1687


,provisional_jurisdiction,jurisdiction_evidence_source,provisional_races,distinct_raw_courses,first_date,last_date,share_of_all_races
0,Great Britain,established_unsuffixed_british_course,76260,55,2015-01-01,2026-05-27,0.403400
1,Great Britain,curated_british_course_configuration,30864,7,2015-01-01,2026-05-27,0.163264
2,Ireland,explicit_terminal_course_code,29215,27,2015-01-01,2025-10-14,0.154542
3,France,explicit_terminal_course_code,19361,73,2015-01-03,2025-10-14,0.102416
4,Hong Kong,explicit_terminal_course_code,6847,2,2015-01-25,2025-10-12,0.036219
5,United States,explicit_terminal_course_code,5863,56,2015-01-01,2025-10-12,0.031014
6,unresolved,cross_jurisdiction_course_name_collision,4534,3,2015-01-03,2026-05-09,0.023984
7,Australia,explicit_terminal_course_code,3798,51,2015-01-01,2025-10-11,0.020091
8,United Arab Emirates,explicit_terminal_course_code,2219,5,2015-01-04,2025-04-12,0.011738
9,Ireland,historical_suffixed_course_link,1568,23,2025-10-15,2026-05-27,0.008294


In [35]:
# Profile the three cross-jurisdiction course-name collisions by period,
# race type and explicit surface evidence.
#
# The earlier suffixed forms establish that:
# - Ascot (AUS)
# - Sandown (AUS)
# - Newcastle (AUS)
#
# are distinct from the longstanding British courses with the same base names.
#
# After the October 2025 suffix-removal change, the unsuffixed raw values may
# represent either jurisdiction. This cell tests whether the ambiguity can be
# separated using source evidence already available at race level:
#
# - date before or after 15 October 2025;
# - race type;
# - explicit Turf/Dirt/synthetic wording in race_name;
# - off-time format as an additional descriptive attribute.
#
# No jurisdiction is assigned yet.

collision_race_profile = refined_jurisdiction_evidence.loc[
    refined_jurisdiction_evidence["course"].isin(
        ["Ascot", "Sandown", "Newcastle"]
    )
].copy()

collision_race_profile["source_period"] = collision_race_profile["date"].map(
    lambda value: (
        "before_2025_10_15"
        if value < "2025-10-15"
        else "from_2025_10_15"
    )
)

# Reuse the explicit surface-family evidence created earlier.
collision_race_profile = collision_race_profile.merge(
    surface_evidence[
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "explicit_surface_family",
            "race_name_has_explicit_surface",
        ]
    ],
    on=["date", "course", "off", "race_name", "type"],
    how="left",
)

# Preserve the raw off-time while distinguishing leading-zero forms such as
# 05:05 from ordinary UK display forms such as 3:15.
collision_race_profile["off_time_format"] = (
    collision_race_profile["off"]
    .astype(str)
    .map(
        lambda value: (
            "leading_zero_24h_style"
            if re.fullmatch(r"0\d:\d{2}", value)
            else "other_raw_format"
        )
    )
)

collision_context_summary = (
    collision_race_profile
    .groupby(
        [
            "course",
            "source_period",
            "type",
            "explicit_surface_family",
            "off_time_format",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        active_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "course",
            "source_period",
            "provisional_races",
            "type",
            "explicit_surface_family",
        ],
        ascending=[True, True, False, True, True],
    )
    .reset_index(drop=True)
)

print(
    "Collision-course provisional races assessed: "
    f"{len(collision_race_profile):,}"
)
print()
print("Counts by raw course:")
print(
    collision_race_profile["course"]
    .value_counts()
    .sort_index()
    .to_string()
)

collision_context_summary

Collision-course provisional races assessed: 4,534

Counts by raw course:
course
Ascot        1837
Newcastle    1010
Sandown      1687


,course,source_period,type,explicit_surface_family,off_time_format,provisional_races,distinct_race_names,active_dates,first_date,last_date
0,Ascot,before_2025_10_15,Flat,no_explicit_race_name_surface,other_raw_format,1216,628,184,2015-04-29,2025-10-04
1,Ascot,before_2025_10_15,Hurdle,no_explicit_race_name_surface,other_raw_format,249,191,77,2015-01-17,2025-03-30
2,Ascot,before_2025_10_15,Chase,no_explicit_race_name_surface,other_raw_format,229,153,77,2015-01-17,2025-03-30
3,Ascot,before_2025_10_15,NH Flat,no_explicit_race_name_surface,other_raw_format,40,30,40,2015-02-14,2025-02-15
4,Ascot,from_2025_10_15,Flat,no_explicit_race_name_surface,other_raw_format,28,28,4,2025-10-18,2026-05-09
5,Ascot,from_2025_10_15,Hurdle,no_explicit_race_name_surface,other_raw_format,27,27,8,2025-11-01,2026-03-29
6,Ascot,from_2025_10_15,Chase,no_explicit_race_name_surface,other_raw_format,23,23,8,2025-11-01,2026-03-29
7,Ascot,from_2025_10_15,Flat,turf,leading_zero_24h_style,19,19,13,2025-11-01,2026-05-02
8,Ascot,from_2025_10_15,NH Flat,no_explicit_race_name_surface,other_raw_format,4,4,4,2025-11-01,2026-02-14
9,Ascot,from_2025_10_15,Flat,turf,other_raw_format,2,2,2,2025-10-18,2026-04-11


In [36]:
# Inspect every post-suffix-change Flat race at the three colliding course names.
#
# This produces the small race-level set that may require either:
# - direct contextual classification from source fields; or
# - external race-result verification.
#
# Useful evidence includes:
# - full race name;
# - off time;
# - going;
# - distance;
# - horse, jockey and trainer nationality context;
# - whether the race name explicitly states a surface.
#
# One representative runner is selected per provisional race, preferring
# the winner where available. No jurisdiction is assigned in this cell.

post_change_collision_flat_sql = f"""
WITH ranked_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        type,
        race_id,
        dist,
        going,
        age,
        class,
        ran,
        horse,
        jockey,
        trainer,
        draw,
        time,
        pos,
        ROW_NUMBER() OVER (
            PARTITION BY
                date,
                course,
                off,
                race_name,
                type
            ORDER BY
                CASE
                    WHEN CAST(pos AS TEXT) = '1' THEN 0
                    ELSE 1
                END,
                rowid
        ) AS row_rank
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND course IN ('Ascot', 'Sandown', 'Newcastle')
      AND type = 'Flat'
      AND date >= '2025-10-15'
)
SELECT
    source_rowid,
    date,
    course,
    off,
    race_name,
    race_id,
    dist,
    going,
    age,
    class,
    ran,
    horse,
    jockey,
    trainer,
    draw,
    time
FROM ranked_rows
WHERE row_rank = 1
ORDER BY
    course,
    date,
    off
"""

post_change_collision_flat_races = pd.read_sql_query(
    post_change_collision_flat_sql,
    connection,
)

post_change_collision_flat_races["explicit_turf_marker"] = (
    post_change_collision_flat_races["race_name"]
    .str.contains(r"\(Turf\)", regex=True, na=False)
)

print(
    "Post-change Flat races at colliding raw course names: "
    f"{len(post_change_collision_flat_races):,}"
)

print()
print("Counts by course and explicit Turf marker:")
display(
    post_change_collision_flat_races
    .groupby(
        ["course", "explicit_turf_marker"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "provisional_races"})
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

post_change_collision_flat_races

Post-change Flat races at colliding raw course names: 59

Counts by course and explicit Turf marker:


,course,explicit_turf_marker,provisional_races
0,Ascot,False,28
1,Ascot,True,21
2,Newcastle,True,3
3,Sandown,False,7


,source_rowid,date,course,off,race_name,race_id,dist,going,age,class,ran,horse,jockey,trainer,draw,time,explicit_turf_marker
0,1751340,2025-10-18,Ascot,10:02,Liquor Barons - Eurythmic Stakes (3yo+) (Turf),906176,7f,Good,6,Class 1,11,Diamond Scene (AUS),Clint Johnston-Porter,Michael Grantham,9,1:23.79,True
1,1751284,2025-10-18,Ascot,12:55,Qipco British Champions Long Distance Cup,900016,2m,Good,7,Class 1,5,Trawlerman (IRE),William Buick,John & Thady Gosden,2,3:22.29,False
2,1751329,2025-10-18,Ascot,13:30,Qipco British Champions Day Two-Year-Old Conditions Stakes (GBB Race),903349,6f,Good,2,Class 2,11,Mission Central (IRE),Christophe Soumillon,A P OBrien,7,1:13.03,False
3,1751310,2025-10-18,Ascot,14:05,Qipco British Champions Sprint Stakes,900018,6f,Good,3,Class 1,19,Powerful Glory (IRE),Jamie Spencer,Richard Fahey,14,1:11.72,False
4,1751289,2025-10-18,Ascot,14:45,Qipco British Champions Fillies & Mares Stakes,900017,1m4f,Good,4,Class 1,10,Kalpana (GB),Colin Keane,Andrew Balding,7,2:32.67,False
5,1751268,2025-10-18,Ascot,15:25,Queen Elizabeth II Stakes (Sponsored By Qipco),900015,1m,Good,5,Class 1,16,Ciceros Gift (GB),Jason Watson,Charles Hills,1,1:38.35,False
6,1751299,2025-10-18,Ascot,16:05,Qipco Champion Stakes,900014,1m2f,Good,4,Class 1,11,Calandagan (IRE),Mickael Barzalona,F-H Graffard,7,2:03.19,False
7,1751248,2025-10-18,Ascot,16:40,Balmoral Handicap (Sponsored By Qipco),902470,1m,Good,3,Class 2,20,Crown Of Oaks (GB),Tom Marquand,William Haggas,23,1:39.43,False
8,1759080,2025-11-01,Ascot,07:05,TABTouch Prince Of Wales Stakes (3yo+) (Turf),907408,5f,Good,5,Class 1,7,Jokers Grin (AUS),Patrick Carbery,Bernie Miller,6,0:58.38,True
9,1759087,2025-11-01,Ascot,08:52,Asian Beau Stakes Handicap) (3yo+) (Turf),907409,7f,Good,5,Class 1,13,Watch Me Rock (AUS),William Pike,Grant & Alana Williams,2,1:23.83,True


In [37]:
# Apply the final candidate jurisdiction derivation at provisional race grain.
#
# This extends the established course-level evidence with narrowly bounded
# race-context rules for the three exact-name collisions:
#
# Ascot:
# - before 15 October 2025: Great Britain;
# - from that date, Flat races explicitly marked "(Turf)" are Australia;
# - all other observed Ascot races remain Great Britain.
#
# Newcastle:
# - before 15 October 2025: Great Britain;
# - from that date, Flat races explicitly marked "(Turf)" are Australia;
# - Hurdle, Chase and NH Flat remain Great Britain.
#
# Sandown:
# - all unsuffixed records in the current source are Great Britain;
# - the Australian venue remains represented separately as "Sandown (AUS)".
#
# Firenze:
# - assigned through a curated venue reference as Italy.
#
# These are candidate canonicalisation rules for the current extract.
# Raw course values remain unchanged and the evidence source is retained.

def derive_candidate_race_jurisdiction(row):
    """Derive candidate jurisdiction and retain the supporting rule."""
    course_name = str(row["course"])
    race_date = str(row["date"])
    race_type = str(row["type"])
    race_name = str(row["race_name"])

    terminal_code = extract_terminal_jurisdiction_code(course_name)

    if terminal_code is not None:
        return pd.Series(
            [
                terminal_code_to_jurisdiction[terminal_code],
                "explicit_terminal_course_code",
            ]
        )

    if course_name in historical_course_to_code:
        historical_code = historical_course_to_code[course_name]

        return pd.Series(
            [
                terminal_code_to_jurisdiction[historical_code],
                "historical_suffixed_course_link",
            ]
        )

    if course_name in curated_british_course_configurations:
        return pd.Series(
            [
                "Great Britain",
                "curated_british_course_configuration",
            ]
        )

    if course_name == "Firenze":
        return pd.Series(
            [
                "Italy",
                "curated_venue_reference",
            ]
        )

    if course_name == "Ascot":
        if (
            race_date >= "2025-10-15"
            and race_type == "Flat"
            and "(Turf)" in race_name
        ):
            return pd.Series(
                [
                    "Australia",
                    "race_context_course_collision_rule",
                ]
            )

        return pd.Series(
            [
                "Great Britain",
                "race_context_course_collision_rule",
            ]
        )

    if course_name == "Newcastle":
        if (
            race_date >= "2025-10-15"
            and race_type == "Flat"
            and "(Turf)" in race_name
        ):
            return pd.Series(
                [
                    "Australia",
                    "race_context_course_collision_rule",
                ]
            )

        return pd.Series(
            [
                "Great Britain",
                "race_context_course_collision_rule",
            ]
        )

    if course_name == "Sandown":
        return pd.Series(
            [
                "Great Britain",
                "race_context_course_collision_rule",
            ]
        )

    if course_name in established_unsuffixed_british_courses:
        return pd.Series(
            [
                "Great Britain",
                "established_unsuffixed_british_course",
            ]
        )

    return pd.Series(
        [
            "unresolved",
            "no_established_jurisdiction_evidence",
        ]
    )


candidate_jurisdiction_evidence = provisional_race_names.copy()

candidate_jurisdiction_evidence[
    ["candidate_jurisdiction", "jurisdiction_evidence_source"]
] = candidate_jurisdiction_evidence.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

candidate_jurisdiction_summary = (
    candidate_jurisdiction_evidence
    .groupby(
        [
            "candidate_jurisdiction",
            "jurisdiction_evidence_source",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_raw_courses=("course", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

candidate_jurisdiction_summary["share_of_all_races"] = (
    candidate_jurisdiction_summary["provisional_races"]
    / len(candidate_jurisdiction_evidence)
)

candidate_jurisdiction_summary = (
    candidate_jurisdiction_summary
    .sort_values(
        ["provisional_races", "candidate_jurisdiction"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

unresolved_candidate_races = candidate_jurisdiction_evidence.loc[
    candidate_jurisdiction_evidence["candidate_jurisdiction"]
    == "unresolved"
]

print(
    "Provisional races assigned a candidate jurisdiction: "
    f"{len(candidate_jurisdiction_evidence) - len(unresolved_candidate_races):,}"
)
print(
    "Candidate jurisdiction coverage: "
    f"{1 - len(unresolved_candidate_races) / len(candidate_jurisdiction_evidence):.2%}"
)
print(
    "Provisional races remaining unresolved: "
    f"{len(unresolved_candidate_races):,}"
)

candidate_jurisdiction_summary

Provisional races assigned a candidate jurisdiction: 189,043
Candidate jurisdiction coverage: 100.00%
Provisional races remaining unresolved: 0


,candidate_jurisdiction,jurisdiction_evidence_source,provisional_races,distinct_raw_courses,first_date,last_date,share_of_all_races
0,Great Britain,established_unsuffixed_british_course,76260,55,2015-01-01,2026-05-27,0.403400
1,Great Britain,curated_british_course_configuration,30864,7,2015-01-01,2026-05-27,0.163264
2,Ireland,explicit_terminal_course_code,29215,27,2015-01-01,2025-10-14,0.154542
3,France,explicit_terminal_course_code,19361,73,2015-01-03,2025-10-14,0.102416
4,Hong Kong,explicit_terminal_course_code,6847,2,2015-01-25,2025-10-12,0.036219
5,United States,explicit_terminal_course_code,5863,56,2015-01-01,2025-10-12,0.031014
6,Great Britain,race_context_course_collision_rule,4510,3,2015-01-03,2026-05-09,0.023857
7,Australia,explicit_terminal_course_code,3798,51,2015-01-01,2025-10-11,0.020091
8,United Arab Emirates,explicit_terminal_course_code,2219,5,2015-01-04,2025-04-12,0.011738
9,Ireland,historical_suffixed_course_link,1568,23,2025-10-15,2026-05-27,0.008294


In [38]:
# Test a candidate canonical venue identity.
#
# Candidate components:
# - candidate jurisdiction derived at race grain;
# - course label with only a recognised terminal jurisdiction code removed.
#
# Examples:
# - "Chantilly (FR)" and later "Chantilly"
#     -> France + Chantilly
# - "Dundalk (AW) (IRE)" and later "Dundalk (AW)"
#     -> Ireland + Dundalk (AW)
# - British Ascot and Australian Ascot
#     -> separated by jurisdiction
#
# Configuration markers such as "(AW)", "(July)" and "(RH)" are retained
# because the notebook has shown that they can carry real source meaning.
#
# This remains a diagnostic candidate, not a final venue dimension.

def remove_terminal_jurisdiction_code(course_name):
    """Remove only a recognised terminal jurisdiction suffix."""
    course_name = str(course_name)

    return re.sub(
        r"\s+\(("
        + "|".join(map(re.escape, terminal_code_to_jurisdiction))
        + r")\)$",
        "",
        course_name,
    ).strip()


candidate_venue_identity = candidate_jurisdiction_evidence.copy()

candidate_venue_identity["candidate_course_label"] = (
    candidate_venue_identity["course"]
    .map(remove_terminal_jurisdiction_code)
)

venue_raw_form_summary = (
    candidate_venue_identity
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        as_index=False,
    )
    .agg(
        raw_course_forms=("course", "nunique"),
        provisional_races=("race_name", "size"),
        active_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

multi_form_candidate_venues = (
    candidate_venue_identity
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ]
    )["course"]
    .agg(
        raw_course_forms="nunique",
        raw_course_values=lambda values: tuple(sorted(set(values))),
    )
    .reset_index()
)

multi_form_candidate_venues = (
    multi_form_candidate_venues.loc[
        multi_form_candidate_venues["raw_course_forms"] > 1
    ]
    .merge(
        venue_raw_form_summary,
        on=[
            "candidate_jurisdiction",
            "candidate_course_label",
            "raw_course_forms",
        ],
        how="left",
    )
    .sort_values(
        [
            "raw_course_forms",
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

jurisdictions_per_course_label = (
    candidate_venue_identity
    .groupby("candidate_course_label")[
        "candidate_jurisdiction"
    ]
    .nunique()
)

cross_jurisdiction_candidate_labels = (
    jurisdictions_per_course_label.loc[
        jurisdictions_per_course_label > 1
    ]
    .sort_values(ascending=False)
)

print(
    "Candidate venue identities: "
    f"{len(venue_raw_form_summary):,}"
)
print(
    "Candidate venues represented by multiple raw course forms: "
    f"{len(multi_form_candidate_venues):,}"
)
print(
    "Course labels occurring in multiple jurisdictions: "
    f"{len(cross_jurisdiction_candidate_labels):,}"
)

print()
print("Cross-jurisdiction candidate course labels:")
print(cross_jurisdiction_candidate_labels.to_string())

multi_form_candidate_venues

Candidate venue identities: 395
Candidate venues represented by multiple raw course forms: 135
Course labels occurring in multiple jurisdictions: 3

Cross-jurisdiction candidate course labels:
candidate_course_label
Ascot        2
Newcastle    2
Sandown      2


,candidate_jurisdiction,candidate_course_label,raw_course_forms,raw_course_values,provisional_races,active_dates,first_date,last_date
0,Hong Kong,Sha Tin,2,"(Sha Tin, Sha Tin (HK))",4543,472,2015-01-25,2026-05-24
1,France,Chantilly,2,"(Chantilly, Chantilly (FR))",4438,533,2015-01-23,2026-05-19
2,France,Deauville,2,"(Deauville, Deauville (FR))",4317,504,2015-01-03,2026-04-08
3,Ireland,Dundalk (AW),2,"(Dundalk (AW), Dundalk (AW) (IRE))",3597,465,2015-01-02,2026-05-26
4,France,Auteuil,2,"(Auteuil, Auteuil (FR))",3356,419,2015-03-01,2026-05-26
5,Hong Kong,Happy Valley,2,"(Happy Valley, Happy Valley (HK))",2938,344,2016-01-06,2026-05-27
6,France,Saint-Cloud,2,"(Saint-Cloud, Saint-Cloud (FR))",2716,330,2015-03-07,2026-05-22
7,France,Longchamp,2,"(Longchamp, Longchamp (FR))",2044,248,2015-04-06,2026-05-24
8,Ireland,Leopardstown,2,"(Leopardstown, Leopardstown (IRE))",1874,256,2015-01-18,2026-05-15
9,Ireland,Curragh,2,"(Curragh, Curragh (IRE))",1823,236,2015-03-29,2026-05-24


In [39]:
# Test whether multiple raw course forms assigned to the same candidate
# venue identity coexist on the same date.
#
# Expected safe pattern:
# - an older suffixed form ends;
# - a later unsuffixed form begins;
# - the two forms do not appear on the same date.
#
# Same-date coexistence would warn that jurisdiction + candidate course label
# has merged values representing distinct tracks, configurations or venues.

same_date_candidate_venue_forms = (
    candidate_venue_identity
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
            "date",
        ],
        as_index=False,
    )
    .agg(
        raw_course_forms=("course", "nunique"),
        raw_course_values=(
            "course",
            lambda values: tuple(sorted(set(values))),
        ),
        provisional_races=("race_name", "size"),
    )
)

same_date_candidate_venue_collisions = (
    same_date_candidate_venue_forms.loc[
        same_date_candidate_venue_forms["raw_course_forms"] > 1
    ]
    .sort_values(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
            "date",
        ]
    )
    .reset_index(drop=True)
)

candidate_venues_with_same_date_forms = (
    same_date_candidate_venue_collisions[
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ]
    ]
    .drop_duplicates()
)

print(
    "Candidate venue identities represented by multiple raw forms: "
    f"{len(multi_form_candidate_venues):,}"
)
print(
    "Candidate venue identities with multiple raw forms on the same date: "
    f"{len(candidate_venues_with_same_date_forms):,}"
)
print(
    "Same-date collision records: "
    f"{len(same_date_candidate_venue_collisions):,}"
)

same_date_candidate_venue_collisions

Candidate venue identities represented by multiple raw forms: 135
Candidate venue identities with multiple raw forms on the same date: 0
Same-date collision records: 0


,candidate_jurisdiction,candidate_course_label,date,raw_course_forms,raw_course_values,provisional_races


### Surface derivation reset

An earlier attempt derived surface from words such as `Turf`, `Dirt`, `Polytrack`, `Tapeta` and `All-Weather` in `race_name`.

Inspection showed that these terms frequently occur in sponsor names, promotions, memorials and series titles rather than describing the racing surface. That derivation was therefore rejected.

From this point onward:

- `race_name` is not used to derive canonical surface;
- only an explicit `(AW)` marker in raw `course` provides direct source-level surface evidence;
- all remaining surface values stay unresolved pending later external enrichment.

In [40]:
# Rebuild the established candidate surface hierarchy, then summarise it by
# candidate jurisdiction and raw race discipline.
#
# Evidence hierarchy:
# 1. explicit race-name surface wording;
# 2. raw course marked "(AW)";
# 3. otherwise unresolved.
#
# No additional surface inference is introduced.

surface_jurisdiction_profile = candidate_jurisdiction_evidence.copy()

def derive_candidate_surface(row):
    """Derive the directly supported race-level surface category."""
    race_name = str(row["race_name"])
    course_name = str(row["course"])

    has_turf = bool(
        re.search(r"\bTurf\b", race_name, flags=re.IGNORECASE)
    )
    has_dirt = bool(
        re.search(r"\bDirt\b", race_name, flags=re.IGNORECASE)
    )
    has_polytrack = bool(
        re.search(r"\bPolytrack\b", race_name, flags=re.IGNORECASE)
    )
    has_tapeta = bool(
        re.search(r"\bTapeta\b", race_name, flags=re.IGNORECASE)
    )
    has_all_weather = bool(
        re.search(
            r"\bAll[- ]Weather(?: Track)?\b",
            race_name,
            flags=re.IGNORECASE,
        )
    )

    if has_turf:
        return pd.Series(
            ["turf", "explicit_race_name_surface"]
        )

    if has_dirt:
        return pd.Series(
            ["dirt", "explicit_race_name_surface"]
        )

    if has_polytrack:
        return pd.Series(
            ["synthetic_polytrack", "explicit_race_name_surface"]
        )

    if has_tapeta:
        return pd.Series(
            ["synthetic_tapeta", "explicit_race_name_surface"]
        )

    if has_all_weather:
        return pd.Series(
            ["other_all_weather", "explicit_race_name_surface"]
        )

    if "(AW)" in course_name:
        return pd.Series(
            [
                "all_weather_unspecified",
                "raw_course_aw_marker",
            ]
        )

    return pd.Series(
        ["unresolved", "no_direct_surface_evidence"]
    )


surface_jurisdiction_profile[
    ["candidate_surface", "surface_evidence_source"]
] = surface_jurisdiction_profile.apply(
    derive_candidate_surface,
    axis=1,
)

# Add the candidate venue label already validated in the previous section.
surface_jurisdiction_profile["candidate_course_label"] = (
    surface_jurisdiction_profile["course"]
    .map(remove_terminal_jurisdiction_code)
)

surface_jurisdiction_summary = (
    surface_jurisdiction_profile
    .groupby(
        [
            "candidate_jurisdiction",
            "type",
            "candidate_surface",
            "surface_evidence_source",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_candidate_venues=(
            "candidate_course_label",
            "nunique",
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

surface_jurisdiction_totals = (
    surface_jurisdiction_profile
    .groupby(
        [
            "candidate_jurisdiction",
            "type",
        ]
    )
    .size()
    .rename("jurisdiction_type_races")
    .reset_index()
)

surface_jurisdiction_summary = (
    surface_jurisdiction_summary
    .merge(
        surface_jurisdiction_totals,
        on=[
            "candidate_jurisdiction",
            "type",
        ],
        how="left",
    )
)

surface_jurisdiction_summary["share_within_jurisdiction_type"] = (
    surface_jurisdiction_summary["provisional_races"]
    / surface_jurisdiction_summary["jurisdiction_type_races"]
)

surface_jurisdiction_summary = (
    surface_jurisdiction_summary
    .sort_values(
        [
            "candidate_jurisdiction",
            "type",
            "provisional_races",
            "candidate_surface",
        ],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)

resolved_surface_races = (
    surface_jurisdiction_profile["candidate_surface"] != "unresolved"
).sum()

print(
    "Provisional races with directly supported surface: "
    f"{resolved_surface_races:,}"
)
print(
    "Direct surface coverage: "
    f"{resolved_surface_races / len(surface_jurisdiction_profile):.2%}"
)
print(
    "Provisional races with unresolved surface: "
    f"{(
        surface_jurisdiction_profile['candidate_surface']
        == 'unresolved'
    ).sum():,}"
)

surface_jurisdiction_summary

Provisional races with directly supported surface: 79,264
Direct surface coverage: 41.93%
Provisional races with unresolved surface: 109,779


,candidate_jurisdiction,type,candidate_surface,surface_evidence_source,provisional_races,distinct_candidate_venues,first_date,last_date,jurisdiction_type_races,share_within_jurisdiction_type
0,Argentina,Flat,dirt,explicit_race_name_surface,224,3,2015-01-18,2026-05-01,437,0.512586
1,Argentina,Flat,turf,explicit_race_name_surface,212,2,2015-01-03,2026-05-25,437,0.485126
2,Argentina,Flat,unresolved,no_direct_surface_evidence,1,1,2025-06-28,2025-06-28,437,0.002288
3,Australia,Flat,turf,explicit_race_name_surface,4039,50,2015-01-09,2026-05-23,4059,0.995073
4,Australia,Flat,unresolved,no_direct_surface_evidence,19,12,2015-01-01,2020-10-03,4059,0.004681
5,Australia,Flat,synthetic_polytrack,explicit_race_name_surface,1,1,2018-05-06,2018-05-06,4059,0.000246
6,Bahrain,Flat,turf,explicit_race_name_surface,150,2,2015-02-27,2026-04-24,151,0.993377
7,Bahrain,Flat,unresolved,no_direct_surface_evidence,1,1,2024-02-02,2024-02-02,151,0.006623
8,Belgium,Chase,turf,explicit_race_name_surface,1,1,2019-08-27,2019-08-27,1,1.000000
9,Belgium,Flat,unresolved,no_direct_surface_evidence,2,1,2017-03-06,2018-04-16,3,0.666667


In [41]:
# Test whether unresolved British and Irish venue/type combinations can safely
# inherit a venue/configuration-level surface default.
#
# For each candidate jurisdiction-qualified course label, this cell reports:
# - unresolved race counts;
# - directly observed surface families elsewhere at that same venue;
# - whether more than one explicit surface family has been observed.
#
# This allows us to distinguish:
# - ordinary turf venues where missing surface text can reasonably inherit turf;
# - all-weather configurations already encoded in raw course;
# - mixed-surface venues where a broad venue-level rule would be unsafe.
#
# No missing surface value is filled in here.

uk_ire_surface_profile = surface_jurisdiction_profile.loc[
    surface_jurisdiction_profile["candidate_jurisdiction"].isin(
        ["Great Britain", "Ireland"]
    )
].copy()

explicit_surface_families_by_venue = (
    uk_ire_surface_profile.loc[
        uk_ire_surface_profile["candidate_surface"] != "unresolved"
    ]
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ]
    )["candidate_surface"]
    .agg(
        explicit_surface_family_count="nunique",
        explicit_surface_families=lambda values: tuple(
            sorted(set(values))
        ),
    )
    .reset_index()
)

unresolved_uk_ire_by_venue = (
    uk_ire_surface_profile.loc[
        uk_ire_surface_profile["candidate_surface"] == "unresolved"
    ]
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
            "type",
        ],
        as_index=False,
    )
    .agg(
        unresolved_races=("race_name", "size"),
        active_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

unresolved_uk_ire_venue_context = (
    unresolved_uk_ire_by_venue
    .merge(
        explicit_surface_families_by_venue,
        on=[
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        how="left",
    )
)

unresolved_uk_ire_venue_context[
    "explicit_surface_family_count"
] = (
    unresolved_uk_ire_venue_context[
        "explicit_surface_family_count"
    ]
    .fillna(0)
    .astype(int)
)

unresolved_uk_ire_venue_context[
    "explicit_surface_families"
] = unresolved_uk_ire_venue_context[
    "explicit_surface_families"
].map(
    lambda value: value if isinstance(value, tuple) else tuple()
)

unresolved_uk_ire_venue_context = (
    unresolved_uk_ire_venue_context
    .sort_values(
        [
            "unresolved_races",
            "candidate_jurisdiction",
            "candidate_course_label",
            "type",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

print(
    "Unresolved British and Irish races assessed: "
    f"{unresolved_uk_ire_venue_context['unresolved_races'].sum():,}"
)
print(
    "Venue/type groups assessed: "
    f"{len(unresolved_uk_ire_venue_context):,}"
)
print(
    "Groups at venues with multiple directly observed surface families: "
    f"{(
        unresolved_uk_ire_venue_context[
            'explicit_surface_family_count'
        ] > 1
    ).sum():,}"
)

unresolved_uk_ire_venue_context

Unresolved British and Irish races assessed: 109,268
Venue/type groups assessed: 259
Groups at venues with multiple directly observed surface families: 7


,candidate_jurisdiction,candidate_course_label,type,unresolved_races,active_dates,first_date,last_date,explicit_surface_family_count,explicit_surface_families
0,Great Britain,Doncaster,Flat,1896,259,2015-03-28,2026-05-16,0,()
1,Great Britain,Windsor,Flat,1882,267,2015-04-13,2026-05-25,1,"(other_all_weather,)"
2,Ireland,Curragh,Flat,1823,236,2015-03-29,2026-05-24,0,()
3,Great Britain,Haydock,Flat,1645,246,2015-04-25,2026-05-23,1,"(turf,)"
4,Great Britain,Nottingham,Flat,1642,230,2015-04-08,2026-05-19,0,()
5,Great Britain,Yarmouth,Flat,1626,230,2015-08-30,2026-05-20,1,"(turf,)"
6,Great Britain,Bath,Flat,1530,216,2015-04-17,2026-05-26,1,"(turf,)"
7,Great Britain,Newmarket,Flat,1502,205,2015-04-15,2026-05-16,1,"(synthetic_polytrack,)"
8,Great Britain,Brighton,Flat,1480,214,2015-04-21,2025-10-16,1,"(turf,)"
9,Great Britain,Beverley,Flat,1444,199,2015-04-15,2026-05-27,0,()


In [42]:
# Inspect suspicious British and Irish non-turf surface classifications.
#
# A race is included when:
# - jurisdiction is Great Britain or Ireland;
# - the derived surface is dirt, Polytrack, Tapeta or other all-weather;
# - the raw course itself does not contain "(AW)".
#
# These are likely to include false positives where a surface word occurs in:
# - a sponsor name;
# - a series or qualifying condition;
# - promotional wording;
# rather than describing the surface of the race.
#
# The complete race name is retained so each classification can be reviewed.
# No surface value is changed in this cell.

suspicious_uk_ire_surface_terms = (
    surface_jurisdiction_profile.loc[
        surface_jurisdiction_profile["candidate_jurisdiction"].isin(
            ["Great Britain", "Ireland"]
        )
        & surface_jurisdiction_profile["candidate_surface"].isin(
            [
                "dirt",
                "synthetic_polytrack",
                "synthetic_tapeta",
                "other_all_weather",
            ]
        )
        & ~surface_jurisdiction_profile["course"].str.contains(
            r"\(AW\)",
            regex=True,
            na=False,
        )
    ]
    [
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "candidate_jurisdiction",
            "candidate_surface",
            "surface_evidence_source",
        ]
    ]
    .sort_values(
        [
            "candidate_surface",
            "course",
            "date",
            "off",
        ]
    )
    .reset_index(drop=True)
)

suspicious_surface_summary = (
    suspicious_uk_ire_surface_terms
    .groupby(
        [
            "candidate_surface",
            "course",
            "type",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "candidate_surface",
            "provisional_races",
            "course",
            "type",
        ],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

print(
    "Suspicious British and Irish non-turf classifications: "
    f"{len(suspicious_uk_ire_surface_terms):,}"
)
print(
    "Distinct race names involved: "
    f"{suspicious_uk_ire_surface_terms['race_name'].nunique():,}"
)

display(suspicious_surface_summary)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

suspicious_uk_ire_surface_terms

Suspicious British and Irish non-turf classifications: 54
Distinct race names involved: 48


,candidate_surface,course,type,provisional_races,distinct_race_names,first_date,last_date
0,dirt,Worcester,Chase,10,7,2015-09-25,2025-09-26
1,other_all_weather,Newcastle,NH Flat,11,11,2019-01-05,2020-02-21
2,other_all_weather,Hexham,Hurdle,7,6,2019-05-11,2022-09-30
3,other_all_weather,Hexham,Chase,5,3,2019-05-04,2022-06-11
4,other_all_weather,Chepstow,Flat,4,4,2016-06-28,2016-08-17
5,other_all_weather,Sedgefield,Hurdle,4,4,2017-12-26,2022-02-24
6,other_all_weather,Hexham,NH Flat,2,2,2019-05-11,2021-05-08
7,other_all_weather,Market Rasen,Chase,2,2,2024-12-26,2025-12-26
8,other_all_weather,Windsor,Flat,2,2,2025-09-08,2025-09-08
9,other_all_weather,Fontwell,Hurdle,1,1,2017-08-17,2017-08-17


,date,course,off,race_name,type,candidate_jurisdiction,candidate_surface,surface_evidence_source
0,2015-09-25,Worcester,2:20,Bob Love King of The Dirt Memorial Beginners Chase,Chase,Great Britain,dirt,explicit_race_name_surface
1,2016-09-23,Worcester,3:30,Bob Love King Of The Dirt Memorial Handicap Chase,Chase,Great Britain,dirt,explicit_race_name_surface
2,2017-09-29,Worcester,3:10,Bob Love King Of The Dirt Memorial Handicap Chase,Chase,Great Britain,dirt,explicit_race_name_surface
3,2018-09-28,Worcester,3:25,Bob Love King Of The Dirt Memorial Mares Handicap Chase (Qualifier),Chase,Great Britain,dirt,explicit_race_name_surface
4,2019-09-27,Worcester,3:10,Bob Love King Of The Dirt Memorial Mares Handicap Chase (Hereford Handicap Chase Series Qual),Chase,Great Britain,dirt,explicit_race_name_surface
5,2021-09-24,Worcester,2:17,Bob Love King Of The Dirt Handicap Chase,Chase,Great Britain,dirt,explicit_race_name_surface
6,2022-09-23,Worcester,1:20,Bob Love King Of The Dirt Novices Chase (GBB Race),Chase,Great Britain,dirt,explicit_race_name_surface
7,2023-09-29,Worcester,2:15,Bob Love King Of The Dirt Novices Chase (GBB Race),Chase,Great Britain,dirt,explicit_race_name_surface
8,2024-09-27,Worcester,2:35,Bob Love King Of The Dirt Novices Chase (GBB Race),Chase,Great Britain,dirt,explicit_race_name_surface
9,2025-09-26,Worcester,2:07,Bob Love King Of The Dirt Novices Handicap Chase,Chase,Great Britain,dirt,explicit_race_name_surface


In [43]:
# Rebuild candidate surface using course/configuration evidence only.
#
# Rules:
# - raw course containing "(AW)" -> all_weather_unspecified;
# - all other races -> unresolved.
#
# Race name is deliberately excluded because inspection showed many false
# positives from sponsors, promotions, memorials and series titles.
#
# This is a conservative baseline for later venue/configuration curation.

surface_course_only_profile = candidate_jurisdiction_evidence.copy()

surface_course_only_profile["candidate_course_label"] = (
    surface_course_only_profile["course"]
    .map(remove_terminal_jurisdiction_code)
)

aw_course_mask = surface_course_only_profile["course"].str.contains(
    r"\(AW\)",
    regex=True,
    na=False,
)

surface_course_only_profile["candidate_surface"] = "unresolved"
surface_course_only_profile.loc[
    aw_course_mask,
    "candidate_surface",
] = "all_weather_unspecified"

surface_course_only_profile[
    "surface_evidence_source"
] = "no_course_level_surface_evidence"

surface_course_only_profile.loc[
    aw_course_mask,
    "surface_evidence_source",
] = "raw_course_aw_marker"

surface_course_only_summary = (
    surface_course_only_profile
    .groupby(
        [
            "candidate_jurisdiction",
            "type",
            "candidate_surface",
            "surface_evidence_source",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_candidate_venues=(
            "candidate_course_label",
            "nunique",
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "candidate_jurisdiction",
            "type",
            "provisional_races",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

resolved_surface_races = (
    surface_course_only_profile["candidate_surface"]
    != "unresolved"
).sum()

print(
    "Provisional races with course-supported surface: "
    f"{resolved_surface_races:,}"
)
print(
    "Course-supported surface coverage: "
    f"{resolved_surface_races / len(surface_course_only_profile):.2%}"
)
print(
    "Provisional races remaining unresolved: "
    f"{(
        surface_course_only_profile['candidate_surface']
        == 'unresolved'
    ).sum():,}"
)

surface_course_only_summary

Provisional races with course-supported surface: 33,023
Course-supported surface coverage: 17.47%
Provisional races remaining unresolved: 156,020


,candidate_jurisdiction,type,candidate_surface,surface_evidence_source,provisional_races,distinct_candidate_venues,first_date,last_date
0,Argentina,Flat,unresolved,no_course_level_surface_evidence,437,3,2015-01-03,2026-05-25
1,Australia,Flat,unresolved,no_course_level_surface_evidence,4059,51,2015-01-01,2026-05-23
2,Bahrain,Flat,unresolved,no_course_level_surface_evidence,151,2,2015-02-27,2026-04-24
3,Belgium,Chase,unresolved,no_course_level_surface_evidence,1,1,2019-08-27,2019-08-27
4,Belgium,Flat,unresolved,no_course_level_surface_evidence,3,2,2017-03-06,2018-04-16
5,Brazil,Flat,unresolved,no_course_level_surface_evidence,289,3,2015-01-20,2026-05-03
6,Canada,Flat,unresolved,no_course_level_surface_evidence,471,4,2015-04-18,2026-05-02
7,Chile,Flat,unresolved,no_course_level_surface_evidence,204,3,2015-01-02,2026-05-02
8,China,Flat,unresolved,no_course_level_surface_evidence,10,1,2015-04-03,2017-04-02
9,Czech Republic,Chase,unresolved,no_course_level_surface_evidence,9,1,2015-10-11,2025-10-12


In [44]:
# Build the candidate venue/configuration inventory required for curated
# surface mapping.
#
# One row is produced for each:
#     candidate jurisdiction + candidate course label
#
# The inventory retains configuration markers such as:
# - (AW)
# - (July)
# - (RH)
# - (Perth)
#
# It reports:
# - raw course forms represented;
# - race and runner volume;
# - disciplines observed;
# - date coverage;
# - whether the source itself explicitly marks the configuration as AW.
#
# This is the appropriate grain for external venue/surface research.
# No additional surface is inferred in this cell.

runner_counts_by_race = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        type,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off,
        race_name,
        type
    """,
    connection,
)

venue_configuration_inventory = (
    surface_course_only_profile
    .merge(
        runner_counts_by_race,
        on=[
            "date",
            "course",
            "off",
            "race_name",
            "type",
        ],
        how="left",
    )
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        as_index=False,
    )
    .agg(
        raw_course_forms=("course", "nunique"),
        raw_course_values=(
            "course",
            lambda values: tuple(sorted(set(values))),
        ),
        provisional_races=("race_name", "size"),
        runner_rows=("runner_rows", "sum"),
        disciplines=(
            "type",
            lambda values: tuple(sorted(set(values))),
        ),
        active_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        source_aw_marked_races=(
            "candidate_surface",
            lambda values: (
                values == "all_weather_unspecified"
            ).sum(),
        ),
    )
)

venue_configuration_inventory["source_aw_marker_present"] = (
    venue_configuration_inventory["source_aw_marked_races"] > 0
)

venue_configuration_inventory["surface_research_status"] = (
    venue_configuration_inventory["source_aw_marker_present"]
    .map(
        {
            True: "source_supports_all_weather",
            False: "requires_curated_reference",
        }
    )
)

venue_configuration_inventory = (
    venue_configuration_inventory
    .sort_values(
        [
            "surface_research_status",
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

print(
    "Candidate venue/configuration identities: "
    f"{len(venue_configuration_inventory):,}"
)
print(
    "Configurations with source AW evidence: "
    f"{venue_configuration_inventory['source_aw_marker_present'].sum():,}"
)
print(
    "Configurations requiring curated surface reference: "
    f"{(~venue_configuration_inventory['source_aw_marker_present']).sum():,}"
)

venue_configuration_inventory

Candidate venue/configuration identities: 395
Configurations with source AW evidence: 7
Configurations requiring curated surface reference: 388


,candidate_jurisdiction,candidate_course_label,raw_course_forms,raw_course_values,provisional_races,runner_rows,disciplines,active_dates,first_date,last_date,source_aw_marked_races,source_aw_marker_present,surface_research_status
0,Hong Kong,Sha Tin,2,"(Sha Tin, Sha Tin (HK))",4543,56963,"(Flat,)",472,2015-01-25,2026-05-24,0,False,requires_curated_reference
1,France,Chantilly,2,"(Chantilly, Chantilly (FR))",4438,52403,"(Flat, NH Flat)",533,2015-01-23,2026-05-19,0,False,requires_curated_reference
2,France,Deauville,2,"(Deauville, Deauville (FR))",4317,52014,"(Flat,)",504,2015-01-03,2026-04-08,0,False,requires_curated_reference
3,France,Auteuil,2,"(Auteuil, Auteuil (FR))",3356,34928,"(Chase, Flat, Hurdle)",419,2015-03-01,2026-05-26,0,False,requires_curated_reference
4,Hong Kong,Happy Valley,2,"(Happy Valley, Happy Valley (HK))",2938,34012,"(Flat,)",344,2016-01-06,2026-05-27,0,False,requires_curated_reference
5,Great Britain,Doncaster,1,"(Doncaster,)",2805,26166,"(Chase, Flat, Hurdle, NH Flat)",389,2015-01-09,2026-05-16,0,False,requires_curated_reference
6,France,Saint-Cloud,2,"(Saint-Cloud, Saint-Cloud (FR))",2716,30288,"(Flat, NH Flat)",330,2015-03-07,2026-05-22,0,False,requires_curated_reference
7,Great Britain,Ayr,1,"(Ayr,)",2325,20921,"(Chase, Flat, Hurdle, NH Flat)",328,2015-01-02,2026-05-20,0,False,requires_curated_reference
8,Great Britain,Chepstow,1,"(Chepstow,)",2318,20804,"(Chase, Flat, Hurdle, NH Flat)",326,2015-01-30,2026-05-21,0,False,requires_curated_reference
9,Great Britain,Haydock,1,"(Haydock,)",2269,19065,"(Chase, Flat, Hurdle, NH Flat)",331,2015-01-17,2026-05-23,0,False,requires_curated_reference


In [45]:
# Prioritise the curated surface-reference backlog by database coverage.
#
# The source directly supports seven all-weather configurations.
# The remaining 388 configurations require external or curated reference data.
#
# This cell ranks the unresolved configurations by provisional race volume and
# calculates cumulative coverage. It shows how many races and runner rows would
# be covered by researching the top:
# - 10
# - 25
# - 50
# - 100
# - 200
# - all remaining configurations
#
# No surface value is inferred or assigned.

surface_reference_backlog = (
    venue_configuration_inventory.loc[
        ~venue_configuration_inventory["source_aw_marker_present"]
    ]
    .sort_values(
        [
            "provisional_races",
            "runner_rows",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

surface_reference_backlog["research_priority"] = (
    surface_reference_backlog.index + 1
)

surface_reference_backlog["cumulative_races"] = (
    surface_reference_backlog["provisional_races"].cumsum()
)

surface_reference_backlog["cumulative_runner_rows"] = (
    surface_reference_backlog["runner_rows"].cumsum()
)

total_backlog_races = surface_reference_backlog[
    "provisional_races"
].sum()

total_backlog_runner_rows = surface_reference_backlog[
    "runner_rows"
].sum()

surface_reference_backlog["cumulative_race_coverage"] = (
    surface_reference_backlog["cumulative_races"]
    / total_backlog_races
)

surface_reference_backlog["cumulative_runner_coverage"] = (
    surface_reference_backlog["cumulative_runner_rows"]
    / total_backlog_runner_rows
)

priority_thresholds = [
    10,
    25,
    50,
    100,
    200,
    len(surface_reference_backlog),
]

priority_coverage_summary = pd.DataFrame(
    [
        {
            "configurations_researched": threshold,
            "provisional_races_covered": (
                surface_reference_backlog
                .head(threshold)["provisional_races"]
                .sum()
            ),
            "share_of_backlog_races": (
                surface_reference_backlog
                .head(threshold)["provisional_races"]
                .sum()
                / total_backlog_races
            ),
            "runner_rows_covered": (
                surface_reference_backlog
                .head(threshold)["runner_rows"]
                .sum()
            ),
            "share_of_backlog_runner_rows": (
                surface_reference_backlog
                .head(threshold)["runner_rows"]
                .sum()
                / total_backlog_runner_rows
            ),
        }
        for threshold in priority_thresholds
    ]
)

print(
    "Venue/configurations requiring curated surface research: "
    f"{len(surface_reference_backlog):,}"
)
print(
    "Provisional races in curated-reference backlog: "
    f"{total_backlog_races:,}"
)
print(
    "Runner rows in curated-reference backlog: "
    f"{total_backlog_runner_rows:,}"
)

display(priority_coverage_summary)

surface_reference_backlog[
    [
        "research_priority",
        "candidate_jurisdiction",
        "candidate_course_label",
        "provisional_races",
        "runner_rows",
        "disciplines",
        "first_date",
        "last_date",
        "cumulative_race_coverage",
        "cumulative_runner_coverage",
    ]
].head(50)

Venue/configurations requiring curated surface research: 388
Provisional races in curated-reference backlog: 156,020
Runner rows in curated-reference backlog: 1,538,077


,configurations_researched,provisional_races_covered,share_of_backlog_races,runner_rows_covered,share_of_backlog_runner_rows
0,10,32025,0.205262,347564,0.225973
1,25,60106,0.385245,623482,0.405365
2,50,96464,0.618280,965959,0.628030
3,100,142215,0.911518,1397155,0.908378
4,200,154656,0.991258,1524558,0.991210
5,388,156020,1.000000,1538077,1.000000


,research_priority,candidate_jurisdiction,candidate_course_label,provisional_races,runner_rows,disciplines,first_date,last_date,cumulative_race_coverage,cumulative_runner_coverage
0,1,Hong Kong,Sha Tin,4543,56963,"(Flat,)",2015-01-25,2026-05-24,0.029118,0.037035
1,2,France,Chantilly,4438,52403,"(Flat, NH Flat)",2015-01-23,2026-05-19,0.057563,0.071106
2,3,France,Deauville,4317,52014,"(Flat,)",2015-01-03,2026-04-08,0.085233,0.104923
3,4,France,Auteuil,3356,34928,"(Chase, Flat, Hurdle)",2015-03-01,2026-05-26,0.106743,0.127632
4,5,Hong Kong,Happy Valley,2938,34012,"(Flat,)",2016-01-06,2026-05-27,0.125574,0.149745
5,6,Great Britain,Doncaster,2805,26166,"(Chase, Flat, Hurdle, NH Flat)",2015-01-09,2026-05-16,0.143552,0.166758
6,7,France,Saint-Cloud,2716,30288,"(Flat, NH Flat)",2015-03-07,2026-05-22,0.160960,0.186450
7,8,Great Britain,Ayr,2325,20921,"(Chase, Flat, Hurdle, NH Flat)",2015-01-02,2026-05-20,0.175862,0.200052
8,9,Great Britain,Chepstow,2318,20804,"(Chase, Flat, Hurdle, NH Flat)",2015-01-30,2026-05-21,0.190719,0.213578
9,10,Great Britain,Haydock,2269,19065,"(Chase, Flat, Hurdle, NH Flat)",2015-01-17,2026-05-23,0.205262,0.225973


In [46]:
# Consolidate Notebook 04's candidate mapping recommendations.
#
# This records the conclusions supported by the profiling work without
# implementing a final production schema.
#
# Distinctions:
# - raw source attributes must always be preserved;
# - candidate derived attributes are supported by current evidence;
# - unresolved attributes require a better external source rather than
#   increasingly speculative inference.

notebook_04_recommendations = pd.DataFrame(
    [
        {
            "area": "Raw course",
            "recommendation": (
                "Preserve the exact raw course value unchanged."
            ),
            "status": "Required source attribute",
            "reason": (
                "Parenthetical elements encode jurisdiction, surface, "
                "configuration, orientation and location context."
            ),
        },
        {
            "area": "Candidate jurisdiction",
            "recommendation": (
                "Derive jurisdiction using explicit terminal codes, "
                "historical suffix links, curated British configurations "
                "and narrowly bounded race-context collision rules."
            ),
            "status": "Candidate derived attribute",
            "reason": (
                "All 189,043 provisional races were assigned while keeping "
                "Ascot, Newcastle and Sandown cross-jurisdiction identities "
                "separate."
            ),
        },
        {
            "area": "Candidate venue identity",
            "recommendation": (
                "Use candidate jurisdiction plus the course label after "
                "removing only a recognised terminal jurisdiction suffix."
            ),
            "status": "Candidate natural identity",
            "reason": (
                "Reduces 528 raw course values to 395 jurisdiction-qualified "
                "venue/configuration identities with no same-date raw-form "
                "collisions."
            ),
        },
        {
            "area": "Course configuration",
            "recommendation": (
                "Retain markers such as (AW), (July), (RH) and (Perth) "
                "within the candidate course label."
            ),
            "status": "Required candidate attribute",
            "reason": (
                "These markers can distinguish track surface, layout, "
                "orientation or geographically duplicated venue names."
            ),
        },
        {
            "area": "Surface from raw course",
            "recommendation": (
                "Assign all_weather_unspecified only when the raw course "
                "explicitly contains (AW)."
            ),
            "status": "Candidate derived attribute",
            "reason": (
                "Provides direct source support for 33,023 races without "
                "using noisy race-name text."
            ),
        },
        {
            "area": "Surface from race name",
            "recommendation": (
                "Do not derive canonical surface from race_name."
            ),
            "status": "Rejected derivation",
            "reason": (
                "Sponsor names, promotions, memorials and series titles "
                "produce repeated false Turf, Dirt, Polytrack and "
                "All-Weather matches."
            ),
        },
        {
            "area": "Remaining surface",
            "recommendation": (
                "Leave surface unresolved unless supplied by a reliable "
                "race-level external reference."
            ),
            "status": "External enrichment required",
            "reason": (
                "156,020 races lack course-level surface evidence, and many "
                "international venues host multiple surfaces, making static "
                "venue-level mapping unsafe."
            ),
        },
        {
            "area": "Raw discipline",
            "recommendation": (
                "Preserve raw type and validate rare conflicts separately."
            ),
            "status": "Required source attribute",
            "reason": (
                "Sixteen explicit NH Flat races conflict with the recorded "
                "type and can be externally verified as a bounded anomaly set."
            ),
        },
    ]
)

notebook_04_recommendations

,area,recommendation,status,reason
0,Raw course,Preserve the exact raw course value unchanged.,Required source attribute,"Parenthetical elements encode jurisdiction, surface, configuration, orientation and location context."
1,Candidate jurisdiction,"Derive jurisdiction using explicit terminal codes, historical suffix links, curated British configurations and narrowly bounded race-context collision rules.",Candidate derived attribute,"All 189,043 provisional races were assigned while keeping Ascot, Newcastle and Sandown cross-jurisdiction identities separate."
2,Candidate venue identity,Use candidate jurisdiction plus the course label after removing only a recognised terminal jurisdiction suffix.,Candidate natural identity,Reduces 528 raw course values to 395 jurisdiction-qualified venue/configuration identities with no same-date raw-form collisions.
3,Course configuration,"Retain markers such as (AW), (July), (RH) and (Perth) within the candidate course label.",Required candidate attribute,"These markers can distinguish track surface, layout, orientation or geographically duplicated venue names."
4,Surface from raw course,Assign all_weather_unspecified only when the raw course explicitly contains (AW).,Candidate derived attribute,"Provides direct source support for 33,023 races without using noisy race-name text."
5,Surface from race name,Do not derive canonical surface from race_name.,Rejected derivation,"Sponsor names, promotions, memorials and series titles produce repeated false Turf, Dirt, Polytrack and All-Weather matches."
6,Remaining surface,Leave surface unresolved unless supplied by a reliable race-level external reference.,External enrichment required,"156,020 races lack course-level surface evidence, and many international venues host multiple surfaces, making static venue-level mapping unsafe."
7,Raw discipline,Preserve raw type and validate rare conflicts separately.,Required source attribute,Sixteen explicit NH Flat races conflict with the recorded type and can be externally verified as a bounded anomaly set.


In [47]:
# Build the quantitative evidence summary for Notebook 04 closeout.
#
# This cell records the principal observed counts supporting the recommendations.
# It does not introduce any new inference.

notebook_04_evidence_summary = pd.DataFrame(
    [
        {
            "finding": "Raw course values",
            "value": 528,
            "unit": "distinct raw values",
            "interpretation": (
                "Raw course is descriptive source data, not a clean venue key."
            ),
        },
        {
            "finding": "Provisional races",
            "value": 189043,
            "unit": "date + course + off groups",
            "interpretation": (
                "Current race-level profiling grain."
            ),
        },
        {
            "finding": "Candidate venue/configuration identities",
            "value": 395,
            "unit": "jurisdiction-qualified identities",
            "interpretation": (
                "Recognised terminal jurisdiction suffixes removed while "
                "configuration markers are retained."
            ),
        },
        {
            "finding": "Candidate venues with multiple raw forms",
            "value": 135,
            "unit": "candidate identities",
            "interpretation": (
                "Mostly suffixed-to-unsuffixed source-format variants."
            ),
        },
        {
            "finding": "Same-date candidate venue collisions",
            "value": 0,
            "unit": "collision records",
            "interpretation": (
                "No evidence that the candidate venue identity merges "
                "coexisting raw venue forms."
            ),
        },
        {
            "finding": "Jurisdiction-assigned races",
            "value": 189043,
            "unit": "provisional races",
            "interpretation": (
                "Every provisional race received a candidate jurisdiction."
            ),
        },
        {
            "finding": "Course-supported surface",
            "value": 33023,
            "unit": "provisional races",
            "interpretation": (
                "Only races whose raw course explicitly contains (AW)."
            ),
        },
        {
            "finding": "Course-supported surface coverage",
            "value": 17.47,
            "unit": "percent of provisional races",
            "interpretation": (
                "Direct source evidence is limited but reliable."
            ),
        },
        {
            "finding": "Unresolved surface",
            "value": 156020,
            "unit": "provisional races",
            "interpretation": (
                "Requires external race-level enrichment rather than "
                "race-name or static venue inference."
            ),
        },
        {
            "finding": "Explicit NH Flat discipline conflicts",
            "value": 16,
            "unit": "provisional races",
            "interpretation": (
                "Small bounded anomaly set for separate external validation."
            ),
        },
    ]
)

notebook_04_evidence_summary

,finding,value,unit,interpretation
0,Raw course values,528.00,distinct raw values,"Raw course is descriptive source data, not a clean venue key."
1,Provisional races,189043.00,date + course + off groups,Current race-level profiling grain.
2,Candidate venue/configuration identities,395.00,jurisdiction-qualified identities,Recognised terminal jurisdiction suffixes removed while configuration markers are retained.
3,Candidate venues with multiple raw forms,135.00,candidate identities,Mostly suffixed-to-unsuffixed source-format variants.
4,Same-date candidate venue collisions,0.00,collision records,No evidence that the candidate venue identity merges coexisting raw venue forms.
5,Jurisdiction-assigned races,189043.00,provisional races,Every provisional race received a candidate jurisdiction.
6,Course-supported surface,33023.00,provisional races,Only races whose raw course explicitly contains (AW).
7,Course-supported surface coverage,17.47,percent of provisional races,Direct source evidence is limited but reliable.
8,Unresolved surface,156020.00,provisional races,Requires external race-level enrichment rather than race-name or static venue inference.
9,Explicit NH Flat discipline conflicts,16.00,provisional races,Small bounded anomaly set for separate external validation.


## Notebook 04 conclusion

### Main conclusion

The source supports a reliable candidate mapping for jurisdiction and venue/configuration identity, but it does not support complete race-surface derivation on its own.

### What the source supports

- Preserve the exact raw `course` value unchanged.
- Derive candidate jurisdiction using:
  - recognised terminal jurisdiction codes;
  - historical links between suffixed and unsuffixed course forms;
  - curated British course/configuration values;
  - narrowly bounded rules for Ascot, Newcastle and Sandown.
- Use:

  `candidate_jurisdiction + candidate_course_label`

  as the candidate venue/configuration identity, where only a recognised terminal jurisdiction suffix is removed.
- Retain meaningful configuration markers such as `(AW)`, `(July)`, `(RH)` and `(Perth)`.
- Assign `all_weather_unspecified` only where the raw course explicitly contains `(AW)`.
- Preserve raw `type` and flag the 16 explicit NH Flat conflicts for separate validation.

### Evidence

- 528 distinct raw course values.
- 189,043 provisional races.
- 395 candidate jurisdiction-qualified venue/configuration identities.
- 135 candidate identities represented by multiple raw course forms.
- 0 same-date collisions between multiple raw forms of the same candidate identity.
- Candidate jurisdiction assigned to all 189,043 provisional races.
- 33,023 races, or 17.47%, have direct course-level all-weather evidence.
- 156,020 races remain unresolved for surface.
- Race-name surface inference was rejected because sponsor names, promotions, memorials and series titles repeatedly produced false matches.

### Interpretation

The candidate jurisdiction and venue/configuration mappings are sufficiently supported for later staging design.

Surface is different. The source explicitly identifies only a small subset of all-weather configurations. The remaining surface values should not be guessed from race names or assigned through a permanent venue-only rule, because some venues use multiple surfaces or change surface over time.

### Practical implication

Notebook 04 should close with:

- raw course preserved;
- candidate jurisdiction derived;
- candidate venue/configuration identity derived;
- explicit `(AW)` evidence retained;
- all other surface values left unresolved in the source-derived layer.

External race-level surface research can be added later as a separate, documented enrichment stage after the source-profiling work is complete.

### Confidence

- **Candidate jurisdiction:** high
- **Candidate venue/configuration identity:** high
- **Surface from explicit `(AW)` marker:** high
- **Surface for all remaining races:** unresolved from this source
- **Raw discipline:** generally reliable, with a small bounded anomaly set

In [49]:
# Validate the principal Notebook 04 conclusions before closeout.
#
# This cell reconstructs the explicit NH Flat conflict set internally so it
# remains reproducible after a clean kernel restart.

assert len(candidate_jurisdiction_evidence) == 189_043

assert candidate_jurisdiction_evidence[
    "candidate_jurisdiction"
].notna().all()

assert len(venue_configuration_inventory) == 395

assert len(multi_form_candidate_venues) == 135

assert len(same_date_candidate_venue_collisions) == 0

assert (
    surface_course_only_profile["candidate_surface"]
    == "all_weather_unspecified"
).sum() == 33_023

assert (
    surface_course_only_profile["candidate_surface"]
    == "unresolved"
).sum() == 156_020

nh_flat_wording_mask = (
    candidate_jurisdiction_evidence["race_name"]
    .str.contains(
        r"\b(?:NHF|NH Flat|National Hunt Flat)\b",
        case=False,
        regex=True,
        na=False,
    )
)

closeout_nh_flat_conflicts = (
    candidate_jurisdiction_evidence.loc[
        nh_flat_wording_mask
        & (
            candidate_jurisdiction_evidence["type"]
            != "NH Flat"
        ),
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
        ],
    ]
    .sort_values(
        [
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)

assert len(closeout_nh_flat_conflicts) == 8

assert (
    surface_course_only_profile["candidate_surface"]
    .value_counts()
    .sum()
    == 189_043
)

print("Notebook 04 validation passed.")
print("Candidate jurisdictions assigned: 189,043 / 189,043")
print("Candidate venue/configuration identities: 395")
print("Candidate venues with multiple raw forms: 135")
print("Same-date venue-form collisions: 0")
print("Course-supported AW races: 33,023")
print("Surface unresolved from source: 156,020")
print("Reproducible explicit NH Flat discipline conflicts: 8")

Notebook 04 validation passed.
Candidate jurisdictions assigned: 189,043 / 189,043
Candidate venue/configuration identities: 395
Candidate venues with multiple raw forms: 135
Same-date venue-form collisions: 0
Course-supported AW races: 33,023
Surface unresolved from source: 156,020
Reproducible explicit NH Flat discipline conflicts: 8


## Validation result

Notebook 04 validation passed.

The final reproducible findings are:

- 189,043 provisional races received a candidate jurisdiction.
- 528 raw course values reduce to 395 jurisdiction-qualified candidate venue/configuration identities.
- 135 candidate identities are represented by multiple raw course forms.
- No candidate identity has multiple raw forms on the same date.
- 33,023 races have direct all-weather evidence from an explicit `(AW)` course marker.
- 156,020 races remain unresolved for surface from the supplied source.
- 8 races contain explicit `NHF`, `NH Flat` or `National Hunt Flat` wording but have another recorded `type`.

The earlier count of 16 discipline conflicts is superseded by the reproducible count of 8 and should be replaced anywhere it appears in the notebook conclusion or evidence summary.

In [50]:
# Write the Notebook 04 closeout record.
#
# This records the final validated findings and the files affected by the
# notebook. It does not modify the raw source database.

from pathlib import Path
import json

closeout_path = (
    PROJECT_ROOT
    / "docs"
    / "NOTEBOOK_04_CLOSEOUT.json"
)

notebook_04_closeout = {
    "notebook": "notebooks/04_course_jurisdiction_and_surface_mapping.ipynb",
    "status": "complete",
    "source_database": (
        "data/raw/form_2015-present/form_2015-present/raceform.db"
    ),
    "source_table": "data",
    "profiling_grain": {
        "provisional_race_key": [
            "date",
            "course",
            "off",
        ],
        "provisional_races": 189043,
    },
    "validated_findings": {
        "raw_course_values": 528,
        "candidate_jurisdictions_assigned": 189043,
        "candidate_venue_configuration_identities": 395,
        "candidate_venues_with_multiple_raw_forms": 135,
        "same_date_candidate_venue_collisions": 0,
        "course_supported_all_weather_races": 33023,
        "course_supported_surface_coverage_pct": 17.47,
        "surface_unresolved_from_source": 156020,
        "explicit_nh_flat_type_conflicts": 8,
    },
    "decisions": [
        "Preserve the exact raw course value.",
        (
            "Derive candidate jurisdiction using recognised terminal codes, "
            "historical suffix links, curated British configurations and "
            "bounded collision rules."
        ),
        (
            "Use candidate jurisdiction plus candidate course label as the "
            "candidate venue/configuration identity."
        ),
        (
            "Remove only recognised terminal jurisdiction suffixes from the "
            "candidate course label."
        ),
        (
            "Retain configuration markers such as (AW), (July), (RH) "
            "and (Perth)."
        ),
        (
            "Assign all_weather_unspecified only where raw course explicitly "
            "contains (AW)."
        ),
        "Do not derive canonical surface from race_name.",
        (
            "Leave remaining surface unresolved until a later documented "
            "external race-level enrichment stage."
        ),
        (
            "Preserve raw type and externally validate the eight reproducible "
            "explicit NH Flat conflicts."
        ),
    ],
    "validation": {
        "clean_kernel_run_all": True,
        "result": "passed",
    },
}

closeout_path.parent.mkdir(parents=True, exist_ok=True)

closeout_path.write_text(
    json.dumps(
        notebook_04_closeout,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Wrote closeout record: {closeout_path}")
print(f"File exists: {closeout_path.exists()}")

Wrote closeout record: /home/rob/Documents/inside-rails-horse-racing/docs/NOTEBOOK_04_CLOSEOUT.json
File exists: True


In [51]:
# Inspect the repository state after completing Notebook 04.
#
# This confirms which files changed before we update project documentation
# and prepare the final commit.

import subprocess

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print("Repository changes:")
print(git_status.stdout or "Working tree clean.")

Repository changes:
?? docs/NOTEBOOK_04_CLOSEOUT.json
?? notebooks/Untitled.ipynb



In [52]:
# Confirm that Notebook 04 now has the intended filename and that no
# accidental Untitled notebook remains.

import subprocess

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print("Repository changes:")
print(git_status.stdout or "Working tree clean.")

Repository changes:
?? docs/NOTEBOOK_04_CLOSEOUT.json
?? notebooks/04_course_jurisdiction_and_surface_mapping.ipynb



In [53]:
# Find project documentation that references Notebook 04, the notebook
# sequence, or the next profiling task.
#
# This identifies the files that need closeout updates without changing them.

import subprocess

documentation_search = subprocess.run(
    [
        "grep",
        "-RInE",
        (
            "Notebook 04|04_course_jurisdiction|"
            "course jurisdiction|surface mapping|"
            "Notebook 05|next notebook"
        ),
        "README.md",
        "docs",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(
    documentation_search.stdout
    or "No matching project-documentation references found."
)

README.md:60:The next bounded study is course, jurisdiction and surface mapping. Final target-schema design remains deferred.
README.md:71:6. update project entry documentation before starting the next notebook.
docs/NOTEBOOK_02_CLOSEOUT.json:39:    "course jurisdiction and surface mapping",
docs/PROJECT_PLAN.md:101:1. course, jurisdiction and surface mapping;
docs/PROJECT_PLAN.md:159:Create Notebook 04 as a bounded study of course, jurisdiction and surface mapping.
docs/NOTEBOOK_04_CLOSEOUT.json:2:  "notebook": "notebooks/04_course_jurisdiction_and_surface_mapping.ipynb",
docs/NOTEBOOK_03_CLOSEOUT.json:76:    "course jurisdiction and surface mapping",
docs/NOTEBOOK_03_CLOSEOUT.json:88:  "next_notebook": "Course, jurisdiction and surface mapping"
docs/NOTEBOOK_01_CLOSEOUT.json:38:    "course jurisdiction and surface",
docs/REPORT_03_RACE_IDENTITY_AND_SOURCE_KEY_RECONSTRUCTION.md:205:The next bounded study should examine course, jurisdiction and surface mapping.



In [54]:
# Display the relevant README and project-plan sections before editing them.

from pathlib import Path

def show_lines(path, start, end):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    print(f"\n--- {path}:{start}-{end} ---")
    for line_number in range(start, min(end, len(lines)) + 1):
        print(f"{line_number:>4}: {lines[line_number - 1]}")

show_lines(PROJECT_ROOT / "README.md", 50, 78)
show_lines(PROJECT_ROOT / "docs" / "PROJECT_PLAN.md", 90, 170)


--- /home/rob/Documents/inside-rails-horse-racing/README.md:50-78 ---
  50: 
  51: - the supplied `race_id` is reused and cannot serve as a unique race key;
  52: - even `date + race_id` collides for eight pairs of genuinely different races;
  53: - `date + course + off` is unique across the current extract and is the leading candidate natural race identity;
  54: - `race_name` should remain attached as a descriptive validation field;
  55: - adding `horse` identifies one unique runner record within every reconstructed race;
  56: - `num` cannot identify an individual runner because it includes zero sentinels and shared betting-entry numbers;
  57: - original SQLite `rowid` should be preserved as source lineage, not business identity;
  58: - later staging tables will require independent surrogate race and runner-record identifiers.
  59: 
  60: The next bounded study is course, jurisdiction and surface mapping. Final target-schema design remains deferred.
  61: 
  62: ## Working meth

In [55]:
# Update the project entry documentation after completing Notebook 04.
#
# Changes:
# - replace the README's "next study" statement with the validated Notebook 04
#   findings and the next bounded study;
# - mark course/jurisdiction/surface mapping complete in the project plan;
# - set Notebook 05 as the current next action.
#
# The raw source database is not modified.

from pathlib import Path

readme_path = PROJECT_ROOT / "README.md"
project_plan_path = PROJECT_ROOT / "docs" / "PROJECT_PLAN.md"

readme_text = readme_path.read_text(encoding="utf-8")

old_readme_text = (
    "The next bounded study is course, jurisdiction and surface mapping. "
    "Final target-schema design remains deferred."
)

new_readme_text = """Notebook 04 established that:

- all 189,043 provisional races can receive a candidate jurisdiction;
- 528 raw course values reduce to 395 jurisdiction-qualified candidate venue/configuration identities;
- recognised terminal jurisdiction suffixes can be removed while retaining meaningful markers such as `(AW)`, `(July)`, `(RH)` and `(Perth)`;
- the 135 candidate identities represented by multiple raw forms have no same-date form collisions;
- 33,023 races have direct all-weather evidence from an explicit `(AW)` course marker;
- `race_name` is not reliable for surface derivation;
- the remaining 156,020 surface values require later external race-level enrichment;
- eight reproducible explicit NH Flat/type conflicts require separate validation.

The next bounded study is finishing position and non-finish outcomes. Final target-schema design remains deferred."""

assert old_readme_text in readme_text

readme_path.write_text(
    readme_text.replace(old_readme_text, new_readme_text),
    encoding="utf-8",
)

project_plan_text = project_plan_path.read_text(encoding="utf-8")

old_sequence = """Current notebook sequence, refined by Notebooks 02 and 03:

1. course, jurisdiction and surface mapping;
2. finishing position and non-finish outcomes;
3. distance parsing;
4. carried-weight parsing;
5. starting-price parsing;
6. prize and currency parsing;
7. race-time parsing;
8. ratings semantics and availability;
9. horse, trainer, jockey and owner identity risks;
10. coupled-entry interpretation where justified."""

new_sequence = """Current notebook sequence, refined by Notebooks 02, 03 and 04:

1. course, jurisdiction and surface mapping — complete;
2. finishing position and non-finish outcomes;
3. distance parsing;
4. carried-weight parsing;
5. starting-price parsing;
6. prize and currency parsing;
7. race-time parsing;
8. ratings semantics and availability;
9. horse, trainer, jockey and owner identity risks;
10. coupled-entry interpretation where justified."""

assert old_sequence in project_plan_text
project_plan_text = project_plan_text.replace(
    old_sequence,
    new_sequence,
)

old_next_action = """## Current next action

Create Notebook 04 as a bounded study of course, jurisdiction and surface mapping.

The notebook should determine:

- how course names encode jurisdiction and course variants;
- whether jurisdiction can be parsed reliably from course text;
- how all-weather, turf and other surface information is represented;
- where course naming varies across time or source jurisdictions;
- which raw and canonical course attributes a later staging layer must preserve.

The study must remain observational and must not begin final target-schema design."""

new_next_action = """## Current next action

Create Notebook 05 as a bounded study of finishing position and non-finish outcomes.

The notebook should determine:

- how numeric finishing positions and non-finish codes are represented;
- how dead heats, disqualifications and amended results appear;
- how `pos`, `ran`, `btn` and `ovr_btn` interact;
- which values are sentinels, source anomalies or genuine racing outcomes;
- which raw and candidate result attributes a later staging layer must preserve.

The study must remain observational and must not begin final target-schema design."""

assert old_next_action in project_plan_text

project_plan_path.write_text(
    project_plan_text.replace(old_next_action, new_next_action),
    encoding="utf-8",
)

print(f"Updated: {readme_path}")
print(f"Updated: {project_plan_path}")

Updated: /home/rob/Documents/inside-rails-horse-racing/README.md
Updated: /home/rob/Documents/inside-rails-horse-racing/docs/PROJECT_PLAN.md


In [56]:
# Review the final Notebook 04 repository changes before committing.

import subprocess

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

git_diff_stat = subprocess.run(
    ["git", "diff", "--stat"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

git_diff_check = subprocess.run(
    ["git", "diff", "--check"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print("Repository changes:")
print(git_status.stdout or "Working tree clean.")

print("\nDiff summary:")
print(git_diff_stat.stdout or "No tracked-file differences.")

print("\nWhitespace/error check:")
print(git_diff_check.stdout or "Passed.")

Repository changes:
 M README.md
 M docs/PROJECT_PLAN.md
?? docs/NOTEBOOK_04_CLOSEOUT.json
?? notebooks/04_course_jurisdiction_and_surface_mapping.ipynb


Diff summary:
 README.md            | 13 ++++++++++++-
 docs/PROJECT_PLAN.md | 16 ++++++++--------
 2 files changed, 20 insertions(+), 9 deletions(-)


Whitespace/error check:
Passed.


In [57]:
# Stage and commit the completed Notebook 04 work.

import subprocess

files_to_commit = [
    "README.md",
    "docs/PROJECT_PLAN.md",
    "docs/NOTEBOOK_04_CLOSEOUT.json",
    "notebooks/04_course_jurisdiction_and_surface_mapping.ipynb",
]

subprocess.run(
    ["git", "add", *files_to_commit],
    cwd=PROJECT_ROOT,
    check=True,
)

commit_result = subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "Complete course jurisdiction and surface mapping",
    ],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(commit_result.stdout)

status_result = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print("Repository status after commit:")
print(status_result.stdout or "Working tree clean.")

[main abfa869] Complete course jurisdiction and surface mapping
 4 files changed, 48986 insertions(+), 9 deletions(-)
 create mode 100644 docs/NOTEBOOK_04_CLOSEOUT.json
 create mode 100644 notebooks/04_course_jurisdiction_and_surface_mapping.ipynb

Repository status after commit:
Working tree clean.


In [58]:
# Push the completed Notebook 04 commit to origin/main and verify alignment.

import subprocess

push_result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(push_result.stdout or push_result.stderr)

verification = subprocess.run(
    [
        "git",
        "status",
        "--short",
        "--branch",
    ],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print("Repository verification:")
print(verification.stdout)

To https://github.com/rjmac22/inside-rails-horse-racing.git
   9182461..abfa869  main -> main

Repository verification:
## main...origin/main

